# WoodblockSynth V5 — Pipeline thực nghiệm và giao diện tích hợp

Bản sửa từ V4: giữ Mitsuba, camera và sáu điều kiện profile × level.
Metric đi kèm là **reference thử nghiệm lấy định hướng từ bản thảo Sơn**, không phải mã/chốt công thức chính thức của Sơn.
Không chạy OCR. Chỉ nhận kết quả CER do nhóm OCR bàn giao.

Chạy lần lượt từ trên xuống. Sửa `Config` cho đường dẫn của bạn. Chạy pilot nhỏ và duyệt `overlays/*_diagnostic.png`
trước khi tăng số ảnh. E_net trống khi thiếu stroke/ROI đã xác thực, hoặc mẫu số gradient chạm sàn.


## Cách sử dụng
Chạy theo thứ tự từ đầu; phần 12.5 nạp `v5_metrics_update.py` trong cùng repo.
Đặt scan ở `input/stl/03284.stl`; sửa `cfg.mesh_path` ở phần 1 khi đổi scan.
Backend so sánh `metrics/backend.py` được lưu riêng với metrics reference; mask proposal vẫn chưa được xác thực.


## 0. Cài thư viện và thư mục outputs

In [ ]:
# CHẠY TRÊN TERMINAL (yêu cầu chọn kernel python 3.14.4)

# python3.14 --version 
# python3.14 -m venv .venv
# source .venv/bin/activate

# python -m pip install -r requirements.txt

# mkdir -p testing_outputs

# THÊM SCAN VÀO input/stl/03284.stl


## 1. Cấu hình
Mọi tham số nằm ở đây. Các dải `(min, max)` là dải ngẫu nhiên hóa — mỗi ảnh ghi lại giá trị đã bốc vào JSON để làm ablation sau này.

In [ ]:
NOTEBOOK_SOURCE_SHA256 = '8506f309e67fbdfdc9b29d3cc58162b288e511bbab55c4944db6da85ebdec54a'
from dataclasses import dataclass, field, asdict
from pathlib import Path
import json, math, time, os, shutil, csv, random
import numpy as np

# Locate this checkout from the notebook working directory (or set WOODBLOCK_PROJECT_ROOT).
def find_project_root():
    override = os.environ.get("WOODBLOCK_PROJECT_ROOT")
    candidates = [Path(override).expanduser()] if override else [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / "v5_metrics_update.py").is_file() and (candidate / "metrics" / "metrics.py").is_file():
            return candidate
    raise FileNotFoundError("Open the notebook inside the repository, or set WOODBLOCK_PROJECT_ROOT to its directory.")

PROJECT_ROOT = find_project_root()
NOTEBOOK_SOURCE_SHA256 = __import__("hashlib").sha256(
    (PROJECT_ROOT / "mitsuba_research_v5_gpu.ipynb").read_bytes()
).hexdigest()

@dataclass
class Config:
    # --- Research V5 + Metrics (Sơn + Duy Lâm) ---
    annotation_index_path: str = ""  # JSON per mesh, normalized UV masks and original-mesh character labels
    metric_backend_path: str = str(PROJECT_ROOT / "metrics" / "backend.py")    # Son's optional Python plugin (API v1)
    experiment_mode: str = "profiles" # "profiles" or "controlled"
    controlled_conditions: list = field(default_factory=list)
    resume_run_id: str = ""           # e.g. metric_pilot_abcd1234
    decimation_max_p95_mm: float = 0.10
    decimation_strict: bool = True
    decimation_voxel_mm: float = 0.10
    decimation_check_samples: int = 10000
    decimation_refinements: int = 1
    fail_on_missing_glyph: bool = True
    
    
    
    
    # --- Chung ---
    seed: int = 20260921
    n_images: int = 20

    #TĂNG RESOLUTION DỰA TRÊN SIZE ẢNH ĐIẸN THOẠI CAMERA 12MP (TRUNG BÌNH) = 4000 X 3000
    width: int = 2048
    height: int = 1536
    
    spp_gpu: int = 256
    spp_cpu: int = 16
    
    #THÊM OUTPUT MỚI ĐỂ SO SÁNH
    output_root: str = str(PROJECT_ROOT / "testing_outputs")  # profile names/ratios updated; separate output avoids mixing old records

    # --- Nguồn mesh: "synthetic" | "hf" | "file" ---
    mesh_source: str = "file"
    hf_repo_id: str = "ll4ma-lab/ycb-fixed-meshes"
    hf_filename: str = "003_cracker_box/poisson/nontextured.ply"
    hf_filenames: list = field(default_factory=list)   # nhiều vật thể một lượt; để trống = chỉ hf_filename
    mesh_path: str = str(PROJECT_ROOT / "input" / "stl" / "03284.stl")                # để trống = tự tìm mọi mesh trong input_dir
    mesh_paths: list = field(default_factory=list)   # được điền tự động ở bước tiền xử lý
    input_dir: str = str(PROJECT_ROOT / "input" / "stl")
    texture_dir: str = str(PROJECT_ROOT / "input" / "wood_texture")
    mesh_exts: tuple = (".stl", ".ply", ".obj")
    decimate_target_faces: int = 6_000_000   # scan gốc ~38 triệu tam giác -> giảm cho vừa GPU
    stream_threshold_faces: int = 20_000_000  # STL lớn hơn -> đọc từng khúc, gom cụm trước (không nạp cả file vào RAM)
    cluster_max_clusters: int = 10_000_000    # số cụm tối đa ở tầng 1 (quyết định kích thước ô gom cụm)
    labels_path: str = ""              # JSON nhãn ngoài (dùng cho mộc bản thật sau này)

    # --- Ván giả (synthetic) ---
    font_url: str = "https://github.com/nomfoundation/font/releases/download/v5.18/NomNaTong-Regular.ttf"
    font_path: str = str(PROJECT_ROOT / "input" / "fonts" / "NomNaTong-Regular.ttf")                # nếu đã có file font thì điền vào đây
    text: str = ("國家社稷山河錦繡春秋日月明光天地人和萬事亨通福祿壽康寧"
                 "大南寔錄正編第一紀卷五十三廣平布政外州匪渠名奠僞稱乾泰年號")
    board_w_mm: float = 300.0          # rộng (trục x) — ván nằm ngang như mộc bản thật
    board_h_mm: float = 200.0          # cao (trục y)
    board_t_mm: float = 20.0           # dày
    n_cols: int = 10
    n_rows: int = 7
    margin_mm: float = 18.0
    relief_mm: float = 2.5             # chữ NỔI cao 2.5 mm
    grid_res_mm: float = 0.25          # độ phân giải lưới chiều cao
    slope_blur_mm: float = 0.35        # làm mềm mép chữ, giả vết dao
    draw_frame: bool = True            # khung viền + đường kẻ cột (cũng nổi)

    # --- Chuẩn hóa tư thế ---
    plate_ratio_thresh: float = 0.05   # trị riêng nhỏ nhất / trị riêng giữa. Ván mộc bản ~0.001-0.01; hộp YCB ~0.14

    # --- Nhãn giả cho vật thể dạng tấm không có nhãn ---
    pseudo_rows: int = 4
    pseudo_cols: int = 6


    #BỎ NHIỄU GAUSS
    #MODIFIED
    # --- Camera ---
    cam_tilt_deg: tuple = (0.0, 40.0)
    cam_roll_deg: tuple = (-6.0, 6.0)
    cam_fov_deg: tuple = (35.0, 60.0)
    cam_fill_frac: tuple = (0.70, 0.95)   # vật thể chiếm bao nhiêu khung (chiều lớn nhất)

    # --- Ánh sáng ---
    light_elev_deg: tuple = (5.0, 50.0)
    light_dist_m: tuple = (0.5, 1.5)
    light_size_m: tuple = (0.05, 0.15)
    fill_ratio: tuple = (0.01, 0.12)
    second_light_prob: float = 0.15

    # --- Lớp mực còn sót ---
    ink_enabled: bool = True
    ink_texel_mm: float = 0.10          # 1 texel = 0,1 mm (trước đây ~0,23-0,42 mm, lệch hai trục)
    ink_max_side: int = 8192            # chặn trên cạnh dài của mặt nạ/texture
    ink_raised_frac: float = 0.6        # ngưỡng độ cao tương đối (trước: 0.5)
    ink_min_blob_mm: float = 0.3        # bỏ đốm nhỏ hơn cỡ này
    ink_erode_px: int = 1               # thu mặt nạ cho mực nằm gọn trong nét (trước: dilate +1)
    ink_edge_sigma_px: float = 1.0      # làm mềm mép mực ~0,1 mm
    ink_strength: tuple = (0.0, 0.05)   # trước: (0, 0.6)
    ink_patch_thresh: tuple = (0.3, 0.7)

    # --- Phơi sáng & cổng chất lượng ---
    exposure_target: tuple = (0.25, 0.55)
    max_saturated_frac: float = 0.01
    min_coverage_plate: float = 0.20
    min_coverage_other: float = 0.08
    min_shading_contrast: float = 0.18    # tương phản của ảnh BÓNG ĐỔ THUẦN trong vùng chữ (đã hiệu chỉnh, xem mục 11)
    light_resample_max: int = 30          # số lần bốc lại đèn nếu tấm đèn lọt vào khung hình
    max_tries: int = 5

    # --- Tăng nét cho OCR (hậu xử lý, ngẫu nhiên hóa nhẹ) ---
    sharpen_enabled: bool = True
    sharpen_amount: tuple = (0.5, 0.8)    # unsharp mask
    sharpen_sigma_px: tuple = (0.8, 1.5)
    sharpen_skip_prob: float = 0.30   # xác suất bỏ qua sharpen ở profile normal

    # --- Hiệu ứng máy ảnh ---
    wb_gain: tuple = (0.95, 1.05)
    vignette: tuple = (0.0, 0.15)
    defocus_sigma_px: tuple = (0.0, 0.30)
    motion_blur_prob: float = 0.05
    noise_a: tuple = (0.0, 0.0015)
    noise_b: tuple = (0.0, 1e-5)
    jpeg_quality: tuple = (90, 98)
    cam_azim_deg: tuple = (225.0, 290.0)   # camera đứng quanh cạnh dài phía -Y (270° = chính giữa)
    board_rot_z_deg: float = 0.0           # 0 hoặc 180: chỉnh sao chữ thuận chiều trong ảnh

    # --- Nhãn ---
    visibility_tol_m: float = 0.003
    patch_px_per_mm: float = 6.0


cfg = Config()

# Pilot small, then inspect masks before expanding. Set n_images here ONCE.
BENCHMARK_MODE = True  # Never reject samples just for low image-quality metrics in research mode.
cfg.n_images = 12
import uuid
OUTPUT_BASE = Path(cfg.output_root)
RUN_ID = cfg.resume_run_id or ("metric_pilot_" + uuid.uuid4().hex[:8])
cfg.output_root = str(OUTPUT_BASE / RUN_ID)
METRICS_OUTPUT_DIR = Path(cfg.output_root)  # All run artifacts are in one root.
print("Run:", RUN_ID)
print("Output:", cfg.output_root)


## 2. Môi trường
Thử `cuda_ad_rgb` → `llvm_ad_rgb` → `scalar_rgb`, render một cảnh nhỏ để chắc variant chạy được thật. Trên CPU, số mẫu/pixel tự hạ xuống.

In [ ]:
import cv2
import trimesh
import mitsuba as mi
import drjit as dr

def pick_variant():
    """Thử cuda -> llvm -> scalar. Chạy một cảnh nhỏ để chắc chắn variant dùng được thật."""
    for v in ("cuda_ad_rgb", "llvm_ad_rgb", "scalar_rgb"):
        try:
            mi.set_variant(v)
            s = mi.load_dict({"type": "scene", "integrator": {"type": "path"},
                              "sensor": {"type": "perspective", "film": {"type": "hdrfilm", "width": 8, "height": 8}},
                              "e": {"type": "constant"}})
            np.array(mi.render(s, spp=1))
            return v
        except Exception as e:
            print(f"[WARN] variant {v} không dùng được: {str(e)[:90]}")
    raise RuntimeError("Không có variant Mitsuba nào chạy được")

VARIANT = pick_variant()
if VARIANT == "scalar_rgb":
    raise RuntimeError("V5 geometry requires LLVM or CUDA; install LLVM/CUDA before rendering.")
SPP = cfg.spp_gpu if VARIANT.startswith("cuda") else cfg.spp_cpu
T = mi.ScalarTransform4f
ROOT = Path(cfg.output_root)
ROOT.mkdir(parents=True, exist_ok=True)
_probe = ROOT / ".write_test"
_probe.write_text("ok"); _probe.unlink()
print("Ghi được vào:", ROOT)
# MỚI
for sub in ("images", "render_images", "masks", "edge_maps", "geometry", "logs", "labels", "patches", "overlays", "assets", "selftest", "rejected"):
    (ROOT / sub).mkdir(parents=True, exist_ok=True)
try:
    import subprocess
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip() or "không có"
except Exception:
    gpu = "không có"
FALLBACKS = []  # mọi lần dùng phương án dự phòng đều ghi lại ở đây
print(f"Mitsuba {mi.__version__} | variant={VARIANT} | GPU={gpu} | spp={SPP}")
if not VARIANT.startswith("cuda"):
    FALLBACKS.append(f"variant={VARIANT} (không có GPU), spp hạ xuống {SPP}")
    print("[WARN] Đang chạy CPU, sẽ chậm. Kiểm tra Settings → Accelerator → GPU.")

## 3. Nạp mesh
Tự đổi mm → m nếu kích thước lớn nhất > 10 (bản scan mộc bản thường tính bằng mm). Cell dưới liệt kê các file mesh trong repo Hugging Face để chọn `hf_filename`.

In [ ]:
def load_mesh_any(path):
    """Nạp mesh bất kỳ, gộp scene thành một mesh, tự đổi mm -> m."""
    m = trimesh.load(path, force="mesh", process=False)
    if not isinstance(m, trimesh.Trimesh) or len(m.faces) == 0:
        raise ValueError(f"Không nạp được mesh tam giác từ {path}")
    ext = float(m.extents.max())
    unit_note = "mét"
    if ext > 10.0:
        m.apply_scale(0.001)
        unit_note = "mm → đã đổi sang mét"
    print(f"Mesh: {len(m.vertices):,} đỉnh, {len(m.faces):,} mặt | đơn vị: {unit_note} | "
          f"kích thước (mm) = {np.round(m.extents * 1000, 2)} | kín: {m.is_watertight}")
    return m

def list_hf_mesh_files(repo_id=None):
    """Liệt kê file mesh trong repo Hugging Face, nhóm theo vật thể, để chọn hf_filename."""
    from huggingface_hub import list_repo_files
    groups = {}
    for f in list_repo_files(repo_id or cfg.hf_repo_id, repo_type="dataset"):
        if Path(f).suffix.lower() in (".ply", ".obj", ".stl", ".glb"):
            groups.setdefault(Path(f).parts[0], []).append(f)
    for k in sorted(groups):
        print(f"{k}: " + ", ".join(sorted(groups[k], key=lambda x: (Path(x).name != "nontextured.ply", x))[:4]))
    return groups

def get_font_path():
    if cfg.font_path and Path(cfg.font_path).exists():
        return cfg.font_path
    dst = ROOT / "assets" / "NomNaTong-Regular.ttf"
    if not dst.exists():
        import urllib.request
        try:
            urllib.request.urlretrieve(cfg.font_url, dst)
        except Exception as e:
            FALLBACKS.append(f"không tải được font Nôm: {e}")
            for c in ("/usr/share/fonts/opentype/noto/NotoSerifCJK-Regular.ttc",
                      "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"):
                if Path(c).exists():
                    print("[WARN] dùng font dự phòng", c)
                    return c
            raise RuntimeError("Không có font CJK nào. Tải NomNaTong lên Kaggle rồi đặt cfg.font_path")
    return str(dst)
def validate_transcript_font(path, text):
    from fontTools.ttLib import TTFont
    font = TTFont(path, fontNumber=0)
    cmap = font.getBestCmap() or {}
    missing = sorted({ch for ch in text if not ch.isspace() and ord(ch) not in cmap})
    font.close()
    if missing and cfg.fail_on_missing_glyph:
        raise ValueError("Font thiếu glyph; không tạo GT sai: " + repr(missing))
    return missing


In [ ]:
if cfg.mesh_source == "hf":
    try:
        _ = list_hf_mesh_files()
    except Exception as e:
        print("[WARN] không liệt kê được repo Hugging Face:", e)

In [ ]:
# MỚI
for d in (cfg.input_dir, cfg.texture_dir):
    for p in sorted(Path(d).rglob("*")):
        if p.is_file():
            print(p, f"{p.stat().st_size / 1e6:.1f} MB")

In [ ]:
# Fail early before allocating a renderer or loading a large mesh.
if cfg.mesh_source == "file":
    if not Path(cfg.input_dir).is_dir():
        raise FileNotFoundError(f"Create the input directory: {cfg.input_dir}")
    if cfg.mesh_path and not Path(cfg.mesh_path).is_file():
        raise FileNotFoundError(f"Place your scan at {cfg.mesh_path}, or edit cfg.mesh_path in section 1.")
print("Mesh:", cfg.mesh_path or cfg.input_dir)


## 3b. Giảm lưới và kiểm tra khoảng cách bề mặt
Renderer giữ variant GPU đã chọn ở cell 2. Giảm lưới và kiểm tra chạy CPU; validator không tạo scene CUDA.
Cần `rtree` và `scipy`: chạy `%pip install rtree scipy` nếu chưa có.
Đo hai chiều tới tam giác gần nhất, lấy mẫu theo diện tích trong ô 24 mm giữa ván; không chứng nhận toàn bộ bề mặt.
Điều kiện đạt: P95 < ngưỡng Config, max trong mẫu ≤ 1 mm, đối chứng identity < 0.0001 mm.
Nếu chưa đạt, thử ô gom cụm nhỏ hơn một lần; số tam giác gửi GPU giữ nguyên. Có thể vẫn thất bại nếu mesh giảm quá mạnh.


In [ ]:
import gc
import subprocess
import sys
import importlib.util
import hashlib

def find_input_meshes():
    root = Path(cfg.input_dir)
    found = sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in cfg.mesh_exts)
    return found

def ram_free_gb():
    try:
        import psutil
        return psutil.virtual_memory().available / 1e9
    except Exception:
        return float("nan")

STL_REC = np.dtype([("n", "<f4", 3), ("v", "<f4", 9), ("a", "<u2")])
KEY_OFF = 1 << 20          # ô lưới có chỉ số trong [-2^20, 2^20): đủ cho vật 100 m ở ô 0,1 mm
KEY_BITS = 21

def stl_memmap(path):
    """Mở STL nhị phân bằng memmap (không đọc vào RAM). None nếu không phải STL nhị phân."""
    size = Path(path).stat().st_size
    with open(path, "rb") as fh:
        fh.read(80)
        n = int(np.frombuffer(fh.read(4), "<u4")[0])
    if 84 + 50 * n != size:
        return None
    return np.memmap(path, dtype=STL_REC, mode="r", offset=84, shape=(n,))

def _keys(v, voxel):
    """Đỉnh (N,3) -> khóa ô lưới int64 (21 bit mỗi trục)."""
    q = np.floor(v / voxel).astype(np.int64) + KEY_OFF
    return (q[:, 0] << (2 * KEY_BITS)) | (q[:, 1] << KEY_BITS) | q[:, 2]

def cluster_stl(path, voxel_mm=None, max_clusters=12_000_000, chunk=6_000_000,
                patch_box=None, log=print):
    """Giảm lưới tầng 1: gom mọi đỉnh trong cùng ô lưới 3D thành một đỉnh (trung bình),
    đọc file theo từng khúc. RAM gần như không phụ thuộc kích thước file.
    patch_box = (c, R, half_mm): đồng thời gom các tam giác GỐC trong ô vuông giữa ván
    để kiểm tra chất lượng sau này (không phải đọc lại file)."""
    mm = stl_memmap(path); n = len(mm)
    # Ước lượng diện tích bề mặt từ mẫu để chọn kích thước ô
    step = max(1, n // 2_000_000)
    s = np.asarray(mm["v"][::step], np.float64).reshape(-1, 3, 3)
    area = 0.5 * np.linalg.norm(np.cross(s[:, 1] - s[:, 0], s[:, 2] - s[:, 0]), axis=1).mean() * n
    auto = float(np.sqrt(area / max_clusters))
    voxel = float(voxel_mm) if voxel_mm is not None else auto
    if voxel <= 0: raise ValueError("voxel must be positive")
    log(f"   diện tích bề mặt ~{area / 100:.0f} cm², ô gom cụm {voxel:.3f} (đơn vị của file)")
    # Lượt 1: tập khóa ô duy nhất
    t = time.time(); uk = []; tot = 0
    for a in range(0, n, chunk):
        v = np.asarray(mm["v"][a:a + chunk], np.float64).reshape(-1, 3)
        k = np.unique(_keys(v, voxel)); uk.append(k); tot += len(k)
        if tot > 60_000_000:
            uk = [np.unique(np.concatenate(uk))]; tot = len(uk[0])
    K = np.unique(np.concatenate(uk)); del uk; gc.collect()
    log(f"   lượt 1: {len(K):,} cụm ({time.time() - t:.0f}s)")
    # Lượt 2: trung bình vị trí mỗi cụm + tam giác nối các cụm khác nhau
    t = time.time(); m = len(K)
    S = np.zeros((m, 3)); C = np.zeros(m); faces = []; patch = []
    for a in range(0, n, chunk):
        v = np.asarray(mm["v"][a:a + chunk], np.float64).reshape(-1, 3)
        idx = np.searchsorted(K, _keys(v, voxel))
        for d in range(3):
            S[:, d] += np.bincount(idx, weights=v[:, d], minlength=m)
        C += np.bincount(idx, minlength=m)
        f = idx.reshape(-1, 3)
        ok = (f[:, 0] != f[:, 1]) & (f[:, 1] != f[:, 2]) & (f[:, 0] != f[:, 2])
        faces.append(f[ok].astype(np.int32))
        if patch_box is not None:
            c, R, half = patch_box
            tri = v.reshape(-1, 3, 3)
            loc = (tri - c) @ R[:2].T                            # (m, 3, 2)
            # chọn tam giác có BẤT KỲ đỉnh nào trong ô (không theo trọng tâm: tam giác rất lớn,
            # ví dụ mặt lưng phẳng, có trọng tâm nằm ngoài ô dù phủ lên ô)
            lo, hi = loc.min(axis=1), loc.max(axis=1)
            sel = ((lo <= half) & (hi >= -half)).all(axis=1)
            patch.append(tri[sel].astype(np.float32))
        del v, idx, f; gc.collect()
    V = S / C[:, None]; del S, C, K
    F = np.concatenate(faces); del faces; gc.collect()
    Fs = np.sort(F, axis=1)
    _, keep = np.unique(np.ascontiguousarray(Fs).view(np.dtype((np.void, 12))).ravel(), return_index=True)
    F = F[np.sort(keep)]; del Fs, keep; gc.collect()
    log(f"   lượt 2: {len(V):,} đỉnh, {len(F):,} tam giác ({time.time() - t:.0f}s)")
    P = np.concatenate(patch).reshape(-1, 3) if patch else None
    return V, F, P, voxel

def sample_frame(path, n_sample=300_000):
    """Khung PCA của ván từ mẫu tam giác: tâm, ma trận hàng (u, v, pháp tuyến)."""
    mm = stl_memmap(path); step = max(1, len(mm) // n_sample)
    s = np.asarray(mm["v"][::step], np.float64).reshape(-1, 3)
    c = s.mean(0); _, _, vt = np.linalg.svd(s - c, full_matrices=False)
    return c, vt, s

def load_stl_fast(path):
    """Đọc STL nhị phân tiết kiệm RAM: đọc thẳng bằng memmap, giữ float32 như trong file,
    gộp đỉnh trùng theo đúng giá trị bit. Trả về (vertices float32, faces int32), hoặc None nếu
    không phải STL nhị phân.
    Lý do: trimesh.load đổi sang float64 và tạo nhiều bản sao khi gộp đỉnh -> với file 1,9 GB
    (38 triệu tam giác) cần hơn 30 GB RAM và làm chết kernel Kaggle. Hàm này cần khoảng 7 GB."""
    size = Path(path).stat().st_size
    with open(path, "rb") as fh:
        fh.read(80)
        n = int(np.frombuffer(fh.read(4), "<u4")[0])
    if 84 + 50 * n != size:
        return None
    rec = np.dtype([("n", "<f4", 3), ("v", "<f4", 9), ("a", "<u2")])
    mm = np.memmap(path, dtype=rec, mode="r", offset=84, shape=(n,))
    tri = np.ascontiguousarray(mm["v"]).reshape(-1, 3)
    del mm
    key = tri.view(np.dtype((np.void, 12))).ravel()          # mỗi đỉnh = 12 byte
    uniq, inv = np.unique(key, return_inverse=True)
    del key, tri; gc.collect()
    verts = uniq.view("<f4").reshape(-1, 3).copy()
    del uniq
    faces = inv.astype(np.int32).reshape(-1, 3)
    del inv; gc.collect()
    return verts, faces


CPU_CHECKER_SOURCE = '"""CPU-only sampled surface-distance validator; no CUDA/Mitsuba import."""\nimport sys,json\nfrom pathlib import Path\nimport numpy as np\nimport trimesh\n\ndef crop_mesh(v,f,c,R,scale,half):\n    v=(np.asarray(v,dtype=np.float64)*scale-c)@R.T\n    # Batch triangle bounds to avoid allocating all triangle coordinates at once.\n    kept=[]\n    for a in range(0,len(f),200000):\n        ff=f[a:a+200000]; t=v[ff][...,:2]\n        keep=((t.min(1)<=half)&(t.max(1)>=-half)).all(1)\n        kept.append(ff[keep])\n    m=trimesh.Trimesh(v,np.concatenate(kept),process=False)\n    m.remove_unreferenced_vertices()\n    return m\n\ndef samples(m,n,inner,rng):\n    points=[]; remaining=n\n    for _ in range(300):\n        p,_=trimesh.sample.sample_surface(m,max(remaining*2,4096),seed=rng)\n        p=p[(np.abs(p[:,:2])<inner).all(1)]\n        points.append(p[:remaining]); remaining-=min(remaining,len(p))\n        if remaining==0: return np.concatenate(points)\n    raise RuntimeError(\'Insufficient surface samples in inner ROI; validation unavailable\')\n\ndef run(req):\n    c=np.asarray(req[\'center\'])*req[\'scale\'];R=np.asarray(req[\'rotation\']);scale=req[\'scale\']\n    data=np.load(req[\'original\']); v=data[\'vertices\'];f=data[\'faces\']\n    original=crop_mesh(v,f,c,R,scale,24.)\n    reduced=trimesh.load(req[\'candidate\'],force=\'mesh\',process=False)\n    reduced=crop_mesh(reduced.vertices,reduced.faces,c,R,scale,24.)\n    if not len(original.faces) or not len(reduced.faces): raise RuntimeError(\'Empty comparison ROI\')\n    rng=np.random.default_rng(20261105); results=[]; baseline=None\n    for source,target in [(original,reduced),(reduced,original)]:\n        p=samples(source,req[\'samples\'],12.,rng)\n        distances=[]\n        for a in range(0,len(p),128):\n            _,d,_=trimesh.proximity.closest_point(target,p[a:a+128])\n            distances.append(d)\n        results.append(np.concatenate(distances))\n    # Same-surface control on a small independent sample.\n    p=samples(original,128,12.,rng)\n    _,d,_=trimesh.proximity.closest_point(original,p)\n    baseline=float(np.max(d))\n    all_d=np.concatenate(results); finite=np.isfinite(all_d)\n    p95=float(np.quantile(all_d[finite],.95)) if finite.any() else None\n    maximum=float(np.max(all_d[finite])) if finite.any() else None\n    ok=bool(finite.all() and baseline<1e-4 and p95<req[\'threshold\'] and maximum<=1.)\n    return dict(status=\'ok\' if ok else \'failed\',checker_version=\'cpu_nearest_triangle_v3\',\n        p95_surface_distance_mm=p95,max_sampled_distance_mm=maximum,\n        fraction_over_1mm=float(np.mean(all_d>1)),missing_fraction=float(np.mean(~finite)),\n        identity_max_mm=baseline,n_samples=len(all_d),threshold_mm=req[\'threshold\'],\n        scope=\'area_weighted_surface_samples_central_24mm_square_not_whole_board\',\n        direction_p95_mm=[float(np.quantile(x,.95)) for x in results])\nif __name__==\'__main__\':\n    req=json.loads(Path(sys.argv[1]).read_text()); out=Path(req[\'report\'])\n    try: report=run(req)\n    except Exception as e:\n        report={\'status\':\'error\',\'checker_version\':\'cpu_nearest_triangle_v3\',\'reason\':str(e)}\n        out.write_text(json.dumps(report,indent=2));raise\n    out.write_text(json.dumps(report,indent=2));print(json.dumps(report))\n'

CHECKER_VERSION = "cpu_nearest_triangle_v3"

def check_candidate(original_path, candidate_path, c, R, scale, tag):
    helper = ROOT / "assets" / "validate_surface_cpu.py"
    helper.write_text(CPU_CHECKER_SOURCE, encoding="utf-8")
    report_path = ROOT / "assets" / f"{tag}_decimation_check.json"
    request = dict(original=str(original_path), candidate=str(candidate_path),
                   center=np.asarray(c).tolist(),rotation=np.asarray(R).tolist(),scale=scale,
                   samples=cfg.decimation_check_samples,threshold=cfg.decimation_max_p95_mm,
                   report=str(report_path))
    request_path=report_path.with_suffix(".request.json")
    request_path.write_text(json.dumps(request),encoding="utf-8")
    print("Kiểm tra khoảng cách trên CPU; renderer giữ", mi.variant(), flush=True)
    proc=subprocess.run([sys.executable,str(helper),str(request_path)],capture_output=True,text=True)
    if proc.returncode:
        raise RuntimeError("CPU validator failed: " + proc.stderr[-2500:])
    report=json.loads(report_path.read_text())
    print("Decimation check:", tag, report, flush=True)
    return report

def decimate_mesh_file(src):
    import fast_simplification as fs
    for dependency in ("rtree", "scipy"):
        if importlib.util.find_spec(dependency) is None:
            raise RuntimeError(f"Thiếu {dependency}. Chạy %pip install rtree scipy trong kernel này rồi chạy lại.")
    dst=ROOT/"assets"/f"{src.stem}_dec.ply"
    provenance=dst.with_suffix(".provenance.json")
    signature=dict(source=str(src.resolve()),size=src.stat().st_size,mtime=src.stat().st_mtime_ns,
        faces=cfg.decimate_target_faces,voxel_mm=cfg.decimation_voxel_mm,
        samples=cfg.decimation_check_samples,threshold=cfg.decimation_max_p95_mm,
        checker=CHECKER_VERSION,refinements=cfg.decimation_refinements)
    report_path=ROOT/"assets"/f"{src.name}_decimation_check.json"
    if dst.exists() and provenance.exists() and report_path.exists():
        try:
            if json.loads(provenance.read_text())==signature and json.loads(report_path.read_text()).get("status")=="ok":
                print("Dùng lại mesh đã đạt kiểm tra:",dst); return str(dst)
        except (ValueError,OSError): pass
    mm=stl_memmap(src) if src.suffix.lower()==".stl" else None
    streaming=mm is not None and len(mm)>cfg.stream_threshold_faces
    del mm
    c,R,samp=sample_frame(src) if streaming else (None,None,None)
    if streaming:
        scale=1. if np.ptp(samp,axis=0).max()>10 else 1000.
        del samp
    else:
        loaded=load_stl_fast(src) if src.suffix.lower()==".stl" else None
        if loaded is None:
            m=trimesh.load(src,force="mesh",process=False);V0,F0=m.vertices,m.faces;del m
        else: V0,F0=loaded
        scale=1. if np.ptp(V0,axis=0).max()>10 else 1000.
        sub=V0[::max(1,len(V0)//100000)].astype(float)
        c=sub.mean(0);_,_,R=np.linalg.svd(sub-c,full_matrices=False);del sub
    original_path=ROOT/"assets"/f"{src.stem}_original_check_patch.npz"
    for attempt in range(cfg.decimation_refinements+1):
        voxel=cfg.decimation_voxel_mm*(.7**attempt)
        print(f"[{src.name}] thử {attempt+1}: voxel {voxel:.4f} mm, target {cfg.decimate_target_faces:,} faces",flush=True)
        if streaming:
            V,F,Pp,_=cluster_stl(src,voxel_mm=voxel/scale,chunk=1000000,
                patch_box=(c,R,24./scale),log=print)
            if Pp is None or not len(Pp): raise RuntimeError("Không lấy được vùng kiểm tra gốc")
            np.savez(original_path,vertices=Pp,faces=np.arange(len(Pp)).reshape(-1,3));del Pp
        else:
            V,F=V0,F0
            # The validator crops in bounded batches in the child process.
            np.savez(original_path,vertices=V0,faces=F0)
        center=V.mean(0,dtype=np.float64)
        if len(F)>cfg.decimate_target_faces:
            v,f=fs.simplify(V.astype(np.float64)-center,F,
                target_reduction=1-cfg.decimate_target_faces/len(F))
            candidate=trimesh.Trimesh(np.asarray(v,dtype=float)+center,f,process=False)
            del v,f
        else: candidate=trimesh.Trimesh(V,F,process=False)
        del V,F;gc.collect()
        pending=ROOT/"assets"/f"{src.stem}_candidate_unvalidated.ply"
        candidate.export(pending);del candidate;gc.collect()
        report=check_candidate(original_path,pending,c,R,scale,src.name)
        if report['status']=='ok':
            pending.replace(dst);provenance.write_text(json.dumps(signature,indent=2))
            return str(dst)
        if not streaming: break
        print("Chưa đạt: thử lại với ô gom cụm nhỏ hơn, không tăng số faces đưa lên GPU.",flush=True)
    raise RuntimeError(f"Mesh {src.name} chưa đạt kiểm tra. Xem {report_path}. "
        "Đã giữ candidate và original_check_patch để kiểm tra lại mà không đọc STL gốc. "
        "Không đưa mesh chưa đạt vào tập thực nghiệm.")

# Re-running this cell must not silently reuse generated paths.
if globals().get("_DECIMATION_GENERATED_PATHS")==cfg.mesh_paths: cfg.mesh_paths=[]
if cfg.mesh_source=="file" and not cfg.mesh_paths:
    srcs=[Path(cfg.mesh_path)] if cfg.mesh_path else find_input_meshes()
    if not srcs: raise RuntimeError(f"Không tìm thấy mesh trong {cfg.input_dir}")
    print("Tìm thấy",len(srcs),"mesh")
    cfg.mesh_paths=[decimate_mesh_file(p) for p in srcs]
    _DECIMATION_GENERATED_PATHS=list(cfg.mesh_paths)
    print("Sẽ render:",cfg.mesh_paths)


In [ ]:
import json

reports = sorted(
    (ROOT / "assets").glob("*_decimation_check.json"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

if not reports:
    print("Chưa tìm thấy báo cáo kiểm tra.")
else:
    report_path = reports[0]
    report = json.loads(report_path.read_text(encoding="utf-8"))

    print("File:", report_path)
    print(json.dumps(report, indent=2, ensure_ascii=False))

## 4. Ván mộc bản giả (dữ liệu kiểm thử có nhãn thật)
Chữ Nôm **nổi** `relief_mm`, **khắc gương**, cột dọc đọc **phải sang trái**, có khung viền và đường kẻ cột. Mép chữ được làm mềm để giả vết dao. Nhãn là hộp mực thật của từng chữ, và hộp từng cột kèm chuỗi chữ theo thứ tự đọc.

In [ ]:
from PIL import Image, ImageDraw, ImageFont

def heightfield_to_mesh(H_m, res_m, W_m, Hh_m, T_m):
    """Lưới chiều cao (hàng 0 = phía trên, y lớn) -> mesh kín: mặt trên + 4 vách + đáy.
    Mặt đáy chỉ có ít đỉnh, giống bản scan thật (mặt khắc dày đặc, mặt sau thưa)."""
    ny, nx = H_m.shape
    xs = -W_m / 2 + (np.arange(nx) + 0.5) * res_m
    ys = Hh_m / 2 - (np.arange(ny) + 0.5) * res_m
    X, Y = np.meshgrid(xs, ys)
    top = np.stack([X.ravel(), Y.ravel(), H_m.ravel()], 1)
    idx = np.arange(nx * ny).reshape(ny, nx)
    v00, v01 = idx[:-1, :-1].ravel(), idx[:-1, 1:].ravel()
    v10, v11 = idx[1:, :-1].ravel(), idx[1:, 1:].ravel()
    f_top = np.concatenate([np.stack([v10, v11, v01], 1), np.stack([v10, v01, v00], 1)])
    # Vòng biên ngược chiều kim đồng hồ khi nhìn từ trên
    loop = np.concatenate([idx[-1, :], idx[::-1, -1][1:], idx[0, ::-1][1:], idx[:, 0][1:-1]])
    n_top = len(top)
    bot = top[loop].copy(); bot[:, 2] = -T_m
    bot_idx = n_top + np.arange(len(loop))
    a, b = loop, np.roll(loop, -1)
    ab, bb = bot_idx, np.roll(bot_idx, -1)
    f_wall = np.concatenate([np.stack([a, ab, bb], 1), np.stack([a, bb, b], 1)])
    c_idx = n_top + len(loop)
    f_bot = np.stack([np.full_like(ab, c_idx), bb, ab], 1)
    V = np.concatenate([top, bot, [[0.0, 0.0, -T_m]]])
    F = np.concatenate([f_top, f_wall, f_bot])
    return trimesh.Trimesh(V, F, process=False)

def make_synthetic_block():
    """Sinh ván mộc bản giả: chữ Nôm NỔI, khắc gương, cột dọc đọc phải sang trái."""
    res = cfg.grid_res_mm
    W, Hh, m = cfg.board_w_mm, cfg.board_h_mm, cfg.margin_mm
    nx, ny = int(round(W / res)), int(round(Hh / res))
    page = Image.new("L", (nx, ny), 0)
    draw = ImageDraw.Draw(page)
    glyph_page = Image.new("L", (nx,ny), 0)
    roi_page = Image.new("L", (nx,ny), 0)
    font_path = get_font_path()
    missing = validate_transcript_font(font_path, cfg.text)
    if missing:
        raise ValueError("Incomplete font transcript: " + repr(missing))
    mm2px = lambda v: v / res
    cell_w = (W - 2 * m) / cfg.n_cols
    cell_h = (Hh - 2 * m) / cfg.n_rows
    font = ImageFont.truetype(font_path, int(mm2px(0.82 * min(cell_w, cell_h))))

    if cfg.draw_frame:   # khung viền và đường kẻ cột, rộng ~0.8 mm
        lw = max(1, int(round(mm2px(0.8))))
        f0 = mm2px(m - 5)
        draw.rectangle([f0, f0, nx - f0, ny - f0], outline=255, width=lw)
        for c in range(1, cfg.n_cols):
            x = mm2px(m + c * cell_w)
            draw.line([(x, f0), (x, ny - f0)], fill=255, width=lw)

    frame_page = np.asarray(page).copy()
    chars, k = [], 0
    for col in range(cfg.n_cols):          # cột 0 = bên PHẢI trang đọc
        for row in range(cfg.n_rows):
            if k >= len(cfg.text):
                break
            ch = cfg.text[k]; k += 1
            cx_mm = W - m - (col + 0.5) * cell_w      # tọa độ trang, gốc góc trên-trái
            cy_mm = m + (row + 0.5) * cell_h
            cw, chh = int(mm2px(cell_w)), int(mm2px(cell_h))
            cell = Image.new("L", (cw, chh), 0)
            d2 = ImageDraw.Draw(cell)
            bb = d2.textbbox((0, 0), ch, font=font)
            d2.text(((cw - (bb[2] - bb[0])) / 2 - bb[0], (chh - (bb[3] - bb[1])) / 2 - bb[1]),
                    ch, font=font, fill=255)
            a = np.array(cell)
            ys_, xs_ = np.nonzero(a > 127)
            if len(xs_) == 0:
                FALLBACKS.append(f"font không có glyph cho '{ch}', bỏ qua")
                continue
            ox, oy = int(mm2px(cx_mm) - cw / 2), int(mm2px(cy_mm) - chh / 2)
            page.paste(Image.fromarray(np.maximum(np.array(page.crop((ox, oy, ox + cw, oy + chh))), a)), (ox, oy))
            glyph_page.paste(cell,(ox,oy))
            ImageDraw.Draw(roi_page).rectangle([max(0,ox-2),max(0,oy-2),min(nx-1,ox+cw+2),min(ny-1,oy+chh+2)],fill=255)
            # hộp mực (mm, tọa độ trang)
            x0, x1 = (ox + xs_.min()) * res, (ox + xs_.max() + 1) * res
            y0, y1 = (oy + ys_.min()) * res, (oy + ys_.max() + 1) * res
            chars.append({"char": ch, "col": col, "row": row, "box_mm": [x0, y0, x1, y1]})

    mask = np.array(page, dtype=np.float32) / 255.0
    mask = mask[:, ::-1]                      # KHẮC GƯƠNG: lật ngang toàn trang
    sig = cfg.slope_blur_mm / res
    if sig > 0:
        mask = cv2.GaussianBlur(mask, (0, 0), sig)
    Hm = (np.clip(mask * 1.6, 0, 1) * cfg.relief_mm / 1000.0).astype(np.float64)
    mesh = heightfield_to_mesh(Hm, res / 1000, W / 1000, Hh / 1000, cfg.board_t_mm / 1000)

    # Nhãn -> tọa độ ván (mét), sau khi lật gương. z = mặt chữ.
    z = cfg.relief_mm / 1000.0
    def page_box_to_quad(b):
        x0, y0, x1, y1 = b
        X0, X1 = sorted([-(x1 - W / 2), -(x0 - W / 2)])       # lật gương
        Y1, Y0 = Hh / 2 - y0, Hh / 2 - y1                     # y hướng lên
        s = 0.001
        # TL, TR, BR, BL theo hình học của mặt ván
        return [[X0 * s, Y1 * s, z], [X1 * s, Y1 * s, z], [X1 * s, Y0 * s, z], [X0 * s, Y0 * s, z]]
    out_chars = [{"char": c["char"], "col": c["col"], "row": c["row"],
                  "quad": page_box_to_quad(c["box_mm"])} for c in chars]
    out_cols = []
    for col in sorted({c["col"] for c in chars}):
        cc = sorted([c for c in chars if c["col"] == col], key=lambda c: c["row"])
        bx = [min(c["box_mm"][0] for c in cc) - 1.0, min(c["box_mm"][1] for c in cc) - 1.0,
              max(c["box_mm"][2] for c in cc) + 1.0, max(c["box_mm"][3] for c in cc) + 1.0]
        out_cols.append({"text": "".join(c["char"] for c in cc), "col": col,
                         "quad": page_box_to_quad(bx)})
    print(f"Ván giả: {nx}x{ny} ô lưới ({res} mm), {len(out_chars)} chữ, {len(out_cols)} cột, "
          f"{len(mesh.vertices):,} đỉnh")
    global SYNTHETIC_MASKS
    stroke = np.asarray(glyph_page,dtype=np.float32)[:,::-1]/255.0
    frame = frame_page.astype(np.float32)[:,::-1]/255.0
    if sig > 0:
        stroke = cv2.GaussianBlur(stroke,(0,0),sig)
        frame = cv2.GaussianBlur(frame,(0,0),sig)
    # Same slope/relief transfer as mesh: top membership at >= 0.5 relative relief.
    SYNTHETIC_MASKS = {"stroke":np.clip(stroke*1.6,0,1)>=.5,
                       "exclusion":cv2.dilate((frame>.03).astype(np.uint8),np.ones((3,3),np.uint8)).astype(bool),
                       "roi":np.asarray(roi_page)[:,::-1]>0,
                       "bounds":np.array([[-W/2000,-Hh/2000,-cfg.board_t_mm/1000],
                                           [W/2000,Hh/2000,cfg.relief_mm/1000]])}
    return mesh, out_chars, out_cols

## 5. Chuẩn hóa tư thế (theo bài Alignment 2024 và SiNoC 2025)
- **Kiểm tra dạng tấm**: trị riêng PCA nhỏ nhất / trị riêng giữa < `plate_ratio_thresh`.
- **Tìm mặt phẳng**: mặt nền là mức có nhiều điểm nhất; chữ nổi nằm về phía trên của nó → xác định chiều pháp tuyến; khớp lại bằng RANSAC + bình phương tối thiểu.
- **k-alignment**: xoay pháp tuyến về +Z bằng công thức Rodrigues.
- **i-alignment**: xoay quanh Z cho hình chữ nhật bao nhỏ nhất, cạnh dài theo x.
- Mặt in (điểm cao nhất) về z = 0.

**Cùng một ma trận 4×4** được áp cho mesh và mọi nhãn — nhờ vậy nhãn luôn bám đúng chữ. Vật thể không dạng tấm chỉ được căn tâm và đặt đáy ở z = 0.

In [ ]:
def apply_M(items, M):
    out = []
    for x in items:
        y = dict(x)
        q = np.asarray(x["quad"], float)
        y["quad"] = (q @ M[:3, :3].T + M[:3, 3]).tolist()
        out.append(y)
    return out

def rot_a_to_b(a, b):
    """Công thức Rodrigues: ma trận xoay đưa vector a về b."""
    a = np.asarray(a, float) / np.linalg.norm(a)
    b = np.asarray(b, float) / np.linalg.norm(b)
    v = np.cross(a, b); c = float(a @ b); s = np.linalg.norm(v)
    if s < 1e-12:
        if c > 0:
            return np.eye(3)
        p = np.eye(3)[np.argmin(np.abs(a))]          # quay 180 độ quanh một trục vuông góc
        ax = np.cross(a, p); ax /= np.linalg.norm(ax)
        return 2 * np.outer(ax, ax) - np.eye(3)
    K = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    return np.eye(3) + K + K @ K * ((1 - c) / s ** 2)

def ransac_plane(P, n_iter=300, thresh=3e-4, rng=None):
    rng = rng or np.random.default_rng(0)
    best, best_in = None, None
    for _ in range(n_iter):
        s = P[rng.choice(len(P), 3, replace=False)]
        n = np.cross(s[1] - s[0], s[2] - s[0]); nn = np.linalg.norm(n)
        if nn < 1e-12:
            continue
        n /= nn
        inl = np.abs((P - s[0]) @ n) < thresh
        if best_in is None or inl.sum() > best_in.sum():
            best, best_in = n, inl
    Q = P[best_in]; c = Q.mean(0)
    n = np.linalg.svd(Q - c, full_matrices=False)[2][2]   # khớp lại bằng bình phương tối thiểu
    return c, n, best_in.mean()

def _envelope(scene, lo, hi, step, from_top):
    """Bắn lưới tia song song trục z lên mesh (đã ở khung PCA), trả về bản đồ độ cao
    nhìn từ trên (from_top) hoặc từ dưới. Ô không chạm: NaN."""
    xs = np.arange(lo[0] + step / 2, hi[0], step); ys = np.arange(hi[1] - step / 2, lo[1], -step)
    X, Y = np.meshgrid(xs, ys); n = X.size
    z0 = hi[2] + 0.01 if from_top else lo[2] - 0.01
    d = -1.0 if from_top else 1.0
    ray = mi.Ray3f(mi.Point3f(X.ravel().copy(), Y.ravel().copy(), np.full(n, z0)),
                   mi.Vector3f(np.zeros(n), np.zeros(n), np.full(n, d)))
    t = np.array(scene.ray_intersect(ray).t).reshape(X.shape)
    H = np.where(np.isfinite(t), z0 + d * t, np.nan)
    return X, Y, H

def _side_detail(H, sigma_cells):
    """Độ chi tiết và độ lệch (skew) của bề mặt nhìn từ một phía.
    H đã đổi dấu sao cho 'hướng về phía người nhìn' là dương."""
    ok = np.isfinite(H)
    if ok.mean() < 0.2:
        return 0.0, 0.0
    F = np.where(ok, H, np.nanmedian(H)).astype(np.float32)
    r = (F - cv2.GaussianBlur(F, (0, 0), sigma_cells))[ok]
    det = float(np.median(np.abs(r)))
    sk = float(np.mean(r ** 3) / (np.std(r) ** 3 + 1e-30))
    return det, sk

def normalize_pose(mesh, relief_raised=True, rng=None):
    """Trả về (mesh_đã_chuẩn_hóa, M 4x4, plate_like, info).
    Dạng tấm: mặt khắc trùng Oxy, cạnh dài theo x, mặt in cao nhất ở z=0.
    Không dạng tấm: chỉ căn tâm XY và đặt đáy ở z=0.

    MỌI thống kê dựa trên DIỆN TÍCH bề mặt, không dựa trên số đỉnh: sau khi giảm lưới,
    đỉnh dồn dày ở mép chữ và thưa ở vùng phẳng, nên đếm đỉnh làm sai hướng (đã tái hiện:
    tấm ván bị lật ngược, 98.8% mặt quay xuống)."""
    rng = rng or np.random.default_rng(0)
    V = np.asarray(mesh.vertices)
    pts, _ = trimesh.sample.sample_surface(mesh, 200_000, seed=int(rng.integers(1 << 31)))
    c0 = pts.mean(0)
    evals, evecs = np.linalg.eigh(np.cov((pts - c0).T))          # tăng dần
    ratio = evals[0] / max(evals[1], 1e-18)
    M = np.eye(4)
    if ratio >= cfg.plate_ratio_thresh:
        M[:3, 3] = [-c0[0], -c0[1], -V[:, 2].min()]
        m2 = mesh.copy(); m2.apply_transform(M)
        return m2, M, False, {"plate_like": False, "eig_ratio": float(ratio)}

    # Khung PCA: x = trục dài, y = trục giữa, z = pháp tuyến (hệ tay phải)
    ex, ey = evecs[:, 2], evecs[:, 1]
    R0 = np.stack([ex, ey, np.cross(ex, ey)])
    P0 = (V - c0) @ R0.T
    tmp = ROOT / "assets" / "_pose_tmp.ply"
    trimesh.Trimesh(P0, mesh.faces, process=False).export(tmp)
    sc = mi.load_dict({"type": "scene", "o": {"type": "ply", "filename": str(tmp)}})
    lo, hi = P0.min(0), P0.max(0)
    step = float(max(hi[0] - lo[0], hi[1] - lo[1]) / 700)
    sig = max(1.0, 0.01 * (hi[0] - lo[0]) / step)
    X, Y, Ht = _envelope(sc, lo, hi, step, from_top=True)
    _, _, Hb = _envelope(sc, lo, hi, step, from_top=False)
    det_t, sk_t = _side_detail(Ht, sig)
    det_b, sk_b = _side_detail(-Hb, sig)
    want = 1.0 if relief_raised else -1.0
    score_t = det_t * (1.0 if sk_t * want > 0 else 0.3)
    score_b = det_b * (1.0 if sk_b * want > 0 else 0.3)
    flip = score_b > score_t
    if flip:                                                     # quay 180° quanh trục x
        R0 = np.diag([1.0, -1.0, -1.0]) @ R0
        H = -Hb[::-1]                                            # nhìn từ phía mới là "trên"
        X, Y = X, -Y[::-1]
    else:
        H = Ht
    # Mặt nền = mức độ cao phổ biến nhất THEO DIỆN TÍCH (mỗi ô lưới cùng diện tích)
    ok = np.isfinite(H); hv = H[ok]
    hist, e = np.histogram(hv, bins=400)
    floor = 0.5 * (e[hist.argmax()] + e[hist.argmax() + 1])
    band = ok & (np.abs(H - floor) < 1.5e-3)
    Q = np.stack([X[band], Y[band], H[band]], 1)
    if len(Q) > 200_000:
        Q = Q[rng.choice(len(Q), 200_000, replace=False)]
    c1, n2, inl = ransac_plane(Q, rng=rng)
    if n2[2] < 0:
        n2 = -n2
    R1 = rot_a_to_b(n2, [0, 0, 1])
    R = R1 @ R0
    P = (V - c0) @ R.T
    hull = cv2.convexHull(P[:, :2].astype(np.float32) * 1000)
    (_, _), (w, h), ang = cv2.minAreaRect(hull)
    a_ = math.radians(ang) + (0 if w >= h else math.pi / 2)
    ca, sa = math.cos(-a_), math.sin(-a_)
    Rz = np.array([[ca, -sa, 0], [sa, ca, 0], [0, 0, 1]])
    R = Rz @ R
    P = (V - c0) @ R.T
    # mặt in = đỉnh cao nhất của bề mặt nhìn từ trên (theo diện tích, không theo đỉnh)
    ztop = float(np.quantile((np.stack([X[ok], Y[ok], H[ok]], 1) @ (Rz @ R1).T)[:, 2], 0.999))
    M[:3, :3] = R
    M[:3, 3] = -R @ c0 - np.array([0, 0, ztop])
    m2 = mesh.copy(); m2.apply_transform(M)
    info = {"plate_like": True, "eig_ratio": float(ratio), "ransac_inlier_frac": float(inl),
            "detail_top_mm": det_t * 1000, "detail_bottom_mm": det_b * 1000,
            "skew_top": sk_t, "skew_bottom": sk_b, "flipped": bool(flip)}
    return m2, M, True, info

def pseudo_grid_labels(mesh):
    """Nhãn lưới giả trên mặt z=0 cho vật thể dạng tấm không có nhãn — chỉ để kiểm tra phép chiếu."""
    V = np.asarray(mesh.vertices)
    x0, y0 = V[:, 0].min(), V[:, 1].min(); x1, y1 = V[:, 0].max(), V[:, 1].max()
    pad = 0.08
    xs = np.linspace(x0 + pad * (x1 - x0), x1 - pad * (x1 - x0), cfg.pseudo_cols + 1)
    ys = np.linspace(y1 - pad * (y1 - y0), y0 + pad * (y1 - y0), cfg.pseudo_rows + 1)
    out = []
    for i in range(cfg.pseudo_rows):
        for j in range(cfg.pseudo_cols):
            out.append({"char": "", "col": j, "row": i, "pseudo": True,
                        "quad": [[xs[j], ys[i], 0], [xs[j + 1], ys[i], 0],
                                 [xs[j + 1], ys[i + 1], 0], [xs[j], ys[i + 1], 0]]})
    return out

def load_external_labels(path):
    d = json.loads(Path(path).read_text(encoding="utf-8"))
    ch = [{"char": x["char"], "col": x.get("col", -1), "row": x.get("row", -1),
           "quad": x["quad_m"]} for x in d.get("chars", [])]
    co = [{"text": x["text"], "col": x.get("col", -1), "quad": x["quad_m"]} for x in d.get("columns", [])]
    return ch, co

def depthmap_bbox_to_quads(bboxes_px, P_depth, surface_z=0.0):
    """Hộp pixel trên depth map trực giao -> tứ giác 3D (mét).
    Quy ước: [px, py, 1] = P_depth @ [x_mm, y_mm, 1], gốc pixel ở góc trên-trái."""
    Pinv = np.linalg.inv(np.asarray(P_depth, float))
    out = []
    for x0, y0, x1, y1 in bboxes_px:
        q = []
        for u, v in ((x0, y0), (x1, y0), (x1, y1), (x0, y1)):
            w = Pinv @ np.array([u, v, 1.0])
            q.append([w[0] / w[2] / 1000.0, w[1] / w[2] / 1000.0, surface_z])
        out.append(q)
    return np.asarray(out)

## 6. Camera — một nguồn sự thật duy nhất
`make_camera` sinh cả ma trận K, R, t (numpy) lẫn cảm biến Mitsuba từ cùng tham số. Khoảng cách tự tính để vật thể chiếm đúng tỉ lệ khung.

**Quy ước pixel:** tọa độ liên tục, mép trên-trái = (0,0), tâm pixel (i,j) = (i+0.5, j+0.5). OpenCV đặt tâm pixel ở số nguyên nên khi vẽ phải trừ 0.5 (lỗi nửa pixel này đã được phát hiện và sửa khi kiểm thử).

In [ ]:
import colorsys

def _norm(v):
    v = np.asarray(v, float); return v / np.linalg.norm(v)

def make_camera(origin, target, up, fov_deg, W, H):
    """Một nguồn sự thật duy nhất cho camera: sinh K, R, t (numpy) và cảm biến Mitsuba.
    Quy ước ảnh: pixel (0,0) ở góc TRÊN-TRÁI, trục x sang phải, y xuống dưới.
    Mitsuba look_at: trục cục bộ +z = hướng nhìn, +x = cross(up, dir) (chỉ sang TRÁI ảnh)."""
    o, tg = np.asarray(origin, float), np.asarray(target, float)
    f = _norm(tg - o)
    left = _norm(np.cross(up, f))
    up2 = np.cross(f, left)
    right, down = -left, -up2
    R = np.stack([right, down, f])            # thế giới -> camera
    t = -R @ o
    fx = (W / 2) / math.tan(math.radians(fov_deg) / 2)
    K = np.array([[fx, 0, W / 2], [0, fx, H / 2], [0, 0, 1.0]])
    return {"K": K, "R": R, "t": t, "origin": o, "target": tg, "up": np.asarray(up, float),
            "fov_deg": float(fov_deg), "W": W, "H": H}

def project(cam, X):
    """X (N,3) mét -> (u, v, khoảng_cách_tia). Tọa độ liên tục: mép trái-trên ảnh = (0,0),
    tâm pixel (i, j) = (i+0.5, j+0.5). Khi vẽ bằng OpenCV phải trừ 0.5."""
    X = np.asarray(X, float).reshape(-1, 3)
    Xc = X @ cam["R"].T + cam["t"]
    z = Xc[:, 2]
    uv = (Xc @ cam["K"].T)[:, :2] / z[:, None]
    dist = np.linalg.norm(X - cam["origin"], axis=1)
    return uv, z, dist

def mi_sensor(cam, spp, rfilter="box"):
    return {"type": "perspective", "fov": cam["fov_deg"], "fov_axis": "x",
            "near_clip": 0.005, "far_clip": 50.0,
            "to_world": T.look_at(origin=cam["origin"].tolist(), target=cam["target"].tolist(),
                                  up=cam["up"].tolist()),
            "film": {"type": "hdrfilm", "width": cam["W"], "height": cam["H"],
                     "pixel_format": "rgb", "rfilter": {"type": rfilter}},
            "sampler": {"type": "independent", "sample_count": spp}}

def bbox_corners(mesh):
    lo, hi = mesh.bounds
    return np.array([[x, y, z] for x in (lo[0], hi[0]) for y in (lo[1], hi[1]) for z in (lo[2], hi[2])])

def sample_camera(mesh, rng, W, H, plate_like, data_profile="normal", detail_level="sharp"):
    """Lấy mẫu camera theo profile. Với ván mộc bản, CLEAN bị khóa nghiêng <= 2 độ."""
    if data_profile not in CAMERA_GEOMETRY:
        raise ValueError(f"Unknown data profile for camera: {data_profile}")

    geom = get_profile_config(data_profile, detail_level)["camera_geometry"]
    if plate_like:
        tilt = rng.uniform(*geom["tilt_deg"])
        if geom["azim_mode"] == "full":
            azim = rng.uniform(0.0, 360.0)
        else:
            azim = rng.uniform(*cfg.cam_azim_deg)
        roll = rng.uniform(*geom["roll_deg"])
    else:
        # Giữ góc camera legacy cho mesh không phải dạng tấm; bộ profile được tối ưu cho mộc bản.
        tilt = rng.uniform(15.0, 65.0)
        azim = rng.uniform(0.0, 360.0)
        roll = rng.uniform(*cfg.cam_roll_deg)
    fov = rng.uniform(*cfg.cam_fov_deg)
    frac = rng.uniform(*cfg.cam_fill_frac)
    lo, hi = mesh.bounds
    ext = hi - lo
    tgt = (lo + hi) / 2 + np.r_[rng.uniform(-0.03, 0.03, 2) * ext[:2], 0]
    if plate_like:
        tgt[2] = 0.0
    corners = bbox_corners(mesh)
    dvec = np.array([math.sin(math.radians(tilt)) * math.cos(math.radians(azim)),
                     math.sin(math.radians(tilt)) * math.sin(math.radians(azim)),
                     math.cos(math.radians(tilt))])
    fwd = -dvec
    # up0 = np.array([0, 1.0, 0])
    # up0 = up0 - fwd * (up0 @ fwd); up0 /= np.linalg.norm(up0)

    a = math.radians(azim)
    up0 = np.array([-math.cos(a), -math.sin(a), 0.0])   # "phía xa camera" nằm ở đỉnh ảnh
    up0 = up0 - fwd * (up0 @ fwd); up0 /= np.linalg.norm(up0)

    r = math.radians(roll)                     # xoay up quanh trục nhìn (Rodrigues)
    up = up0 * math.cos(r) + np.cross(fwd, up0) * math.sin(r)
    d = np.linalg.norm(ext) * 1.2
    for _ in range(6):
        cam = make_camera(tgt + d * dvec, tgt, up, fov, W, H)
        uv, z, _ = project(cam, corners)
        span = max((uv[:, 0].max() - uv[:, 0].min()) / W, (uv[:, 1].max() - uv[:, 1].min()) / H)
        d *= span / frac
    cam = make_camera(tgt + d * dvec, tgt, up, fov, W, H)
    uv, z, _ = project(cam, corners)
    params = {"tilt_deg": tilt, "azim_deg": azim, "roll_deg": roll, "fov_deg": fov,
              "fill_frac_target": frac, "distance_m": d,
              "all_corners_in_frame": bool((uv[:, 0] >= 0).all() and (uv[:, 0] <= W).all()
                                           and (uv[:, 1] >= 0).all() and (uv[:, 1] <= H).all()
                                           and (z > 0).all())}
    return cam, params

## 7. Vật liệu gỗ
Mesh scan không có màu, nên vân gỗ được sinh thủ tục (vòng năm + nhiễu fractal theo thớ), ngẫu nhiên màu, độ tối, hướng thớ, độ nhám. Với vật thể dạng tấm có thể phủ **mực còn sót** lên mặt chữ nổi.

In [ ]:
_OCC_CACHE = {}
def occlusion_scene(ply):
    cache_key=(str(ply),Path(ply).stat().st_mtime_ns,Path(ply).stat().st_size)
    if cache_key not in _OCC_CACHE:
        _OCC_CACHE[cache_key] = mi.load_dict({"type": "scene", "obj": {"type": "ply", "filename": str(ply)}})
    return _OCC_CACHE[cache_key]

def fractal_noise(rng, S, octaves=6, stretch=1.0):
    acc = np.zeros((S, S), np.float32); amp, tot = 1.0, 0.0
    for o in range(octaves):
        gy = max(2, int(2 ** (o + 1)))
        gx = max(2, int(gy / stretch))
        acc += amp * cv2.resize(rng.random((gy, gx)).astype(np.float32), (S, S), interpolation=cv2.INTER_CUBIC)
        tot += amp; amp *= 0.55
    acc /= tot
    return (acc - acc.min()) / (np.ptp(acc) + 1e-9)

def make_wood_texture(rng, S=1024):
    """Vân gỗ thủ tục cho ván in đã dùng lâu năm:
    - thớ mịn kéo dài theo một hướng là thành phần CHỦ ĐẠO;
    - vòng năm thưa, mảnh (đường gỗ muộn), biên độ nhỏ — bản trước dùng sóng sin đậm
      nên ra các sọc đều như tôn lượn sóng;
    - mảng loang màu tần số thấp;
    - lớp mực ngấm loang lổ làm ván sẫm, như ván đã in nhiều lần."""
    hue = rng.uniform(0.04, 0.09); sat = rng.uniform(0.30, 0.60); val = rng.uniform(0.14, 0.45)
    base = np.array(colorsys.hsv_to_rgb(hue, sat, val), np.float32)
    fine = fractal_noise(rng, S, octaves=7, stretch=rng.uniform(20, 60))
    warp = fractal_noise(rng, S, octaves=4)
    yy = np.linspace(0, 1, S, dtype=np.float32)[:, None]
    freq = rng.uniform(4, 14)
    rings = 0.5 + 0.5 * np.sin(2 * np.pi * (freq * yy + rng.uniform(0.3, 1.2) * warp))
    sharp = rng.uniform(3, 8)
    rings = rings ** sharp                                   # chỉ còn vạch mảnh, không còn sọc
    stain = fractal_noise(rng, S, octaves=3)
    a_f, a_r, a_s = rng.uniform(0.08, 0.20), rng.uniform(0.03, 0.10), rng.uniform(0.10, 0.30)
    mult = 1 + 2 * a_f * (fine - 0.5) - a_r * rings + 2 * a_s * (stain - 0.5)
    ink = rng.uniform(0.0, 0.55)
    wash = ink * np.clip(fractal_noise(rng, S, octaves=3) * 1.4 - 0.2, 0, 1)
    img = base[None, None, :] * np.clip(mult, 0.3, 1.8)[..., None] * (1 - wash)[..., None]
    ang = rng.uniform(0, 180)
    Mrot = cv2.getRotationMatrix2D((S / 2, S / 2), ang, 1.0)
    img = cv2.warpAffine(img, Mrot, (S, S), borderMode=cv2.BORDER_REFLECT)
    info = {"hue": hue, "sat": sat, "val": val, "ring_freq": freq, "ring_sharpness": sharp,
            "grain_amp": a_f, "ring_amp": a_r, "stain_amp": a_s, "ink_wash": ink, "grain_angle_deg": ang}
    return np.clip(img, 0, 1), info

def raised_mask_uv(mesh, ply_path, texel_mm=None, max_side=None):
    """Mặt nạ vùng NỔI, độ phân giải đẳng hướng ~texel_mm, kích thước (Hm, Wm) theo tỉ lệ ván."""
    texel_mm = texel_mm or cfg.ink_texel_mm
    max_side = max_side or cfg.ink_max_side
    lo, hi = mesh.bounds
    ext = hi - lo                                              # mét
    Wm = int(min(max_side, math.ceil(ext[0] * 1000 / texel_mm)))
    Hm = int(min(max_side, math.ceil(ext[1] * 1000 / texel_mm)))
    xs = lo[0] + (np.arange(Wm) + 0.5) / Wm * ext[0]
    ys = hi[1] - (np.arange(Hm) + 0.5) / Hm * ext[1]           # hàng 0 = y lớn (v = 1)
    sc = occlusion_scene(ply_path)
    z = np.full((Hm, Wm), np.nan, np.float32)
    step = max(1, 2_000_000 // Wm)                             # ~2 triệu tia mỗi lượt
    for r0 in range(0, Hm, step):
        r1 = min(Hm, r0 + step)
        X, Y = np.meshgrid(xs, ys[r0:r1]); n = X.size
        ray = mi.Ray3f(mi.Point3f(X.ravel().copy(), Y.ravel().copy(), np.full(n, hi[2] + 0.05)),
                       mi.Vector3f(np.zeros(n), np.zeros(n), -np.ones(n)))
        t = np.array(sc.ray_intersect(ray).t).reshape(r1 - r0, Wm)
        z[r0:r1] = np.where(np.isfinite(t), hi[2] + 0.05 - t, np.nan)
    zt = z[np.isfinite(z) & (z > -0.01)]
    hist, e = np.histogram(zt, bins=200)
    floor = 0.5 * (e[hist.argmax()] + e[hist.argmax() + 1])
    with np.errstate(invalid="ignore"):
        m = ((z - floor) / max(-floor, 1e-4) > cfg.ink_raised_frac).astype(np.uint8)
    # bỏ đốm nhỏ (nhiễu, mảng nền vượt ngưỡng)
    n_lab, lab, st, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    min_area = max(1, int((cfg.ink_min_blob_mm / texel_mm) ** 2))
    keep = np.zeros(n_lab, bool); keep[1:] = st[1:, cv2.CC_STAT_AREA] >= min_area
    m = keep[lab].astype(np.float32)
    if cfg.ink_erode_px > 0:                                   # THU, không nở
        m = cv2.erode(m, np.ones((3, 3), np.uint8), iterations=cfg.ink_erode_px)
    return m

def write_ply_uv(mesh, path):
    """Ghi PLY nhị phân kèm UV phẳng — Mitsuba đọc được trực tiếp."""
    V = np.asarray(mesh.vertices, np.float32); F = np.asarray(mesh.faces, np.int32)
    lo, hi = mesh.bounds
    u = (V[:, 0] - lo[0]) / (hi[0] - lo[0]); v = (V[:, 1] - lo[1]) / (hi[1] - lo[1])
    vert = np.zeros(len(V), dtype=[("x", "<f4"), ("y", "<f4"), ("z", "<f4"), ("u", "<f4"), ("v", "<f4")])
    vert["x"], vert["y"], vert["z"], vert["u"], vert["v"] = V[:, 0], V[:, 1], V[:, 2], u, v
    face = np.zeros(len(F), dtype=[("n", "u1"), ("i", "<i4", (3,))])
    face["n"] = 3; face["i"] = F
    hdr = ("ply\nformat binary_little_endian 1.0\n"
           f"element vertex {len(V)}\nproperty float x\nproperty float y\nproperty float z\n"
           "property float u\nproperty float v\n"
           f"element face {len(F)}\nproperty list uchar int vertex_indices\nend_header\n")
    with open(path, "wb") as fh:
        fh.write(hdr.encode()); fh.write(vert.tobytes()); fh.write(face.tobytes())

## 8. Ánh sáng và cảnh
- **Đèn chính**: đèn vùng chữ nhật, độ cao `light_elev_deg` (thấp = chiếu xiên, bóng dài) — biến quan trọng nhất cho ablation.
- **Ánh sáng môi trường** yếu, tỉ lệ ngẫu nhiên; **đèn phụ** với xác suất 0.3.
- **Nền**: trắng (xác suất 0.4, giống bộ ảnh thật trong bài KSE 2026), xám, vải tối, mặt bàn gỗ.

Lưu ý phát hiện khi kiểm thử: AOV `depth` của Mitsuba đo từ mặt phẳng `near_clip`, **không** từ tâm camera — nên chỉ dùng AOV để lấy mặt nạ vật thể, còn kiểm tra che khuất thì tự bắn tia.

In [ ]:
def rect_emitter(pos, tgt, size, radiance):
    dirv = _norm(np.asarray(tgt) - pos)
    upl = [0, 1, 0] if abs(dirv[2]) > 0.99 else [0, 0, 1]
    return {"type": "rectangle",
            "to_world": T.look_at(origin=pos.tolist(), target=list(map(float, tgt)), up=upl) @ T.scale([size / 2, size / 2, 1]),
            "emitter": {"type": "area", "radiance": {"type": "rgb", "value": radiance}}}
    
REF_DIAG = 0.49   # đường chéo ván mộc bản tham chiếu 0.43×0.24×0.02 m
def light_scale(mesh):
    return float(np.linalg.norm(mesh.extents)) / REF_DIAG
    
def sample_lights(rng, tgt, scale=1.0, data_profile="normal", cam=None, detail_level="sharp"):
    """Lấy đèn theo profile.

    CLEAN dùng đèn raking có cao độ thấp so với mặt ván để làm nổi relief chữ.
    NORMAL/HARD dùng lấy mẫu ngẫu nhiên theo cao độ và phương vị; sample_lights_outside
    vẫn loại đèn nào lọt vào khung hình camera.
    """
    if data_profile not in LIGHT_PROFILE:
        raise ValueError(f"Unknown data profile for lighting: {data_profile}")
    lr = get_profile_config(data_profile, detail_level)["lighting"]
    tgt = np.asarray(tgt, dtype=float)
    size = rng.uniform(*lr["size_ratio"]) * scale
    power_mult = float(rng.uniform(*lr["key_power_multiplier"]))
    placement_mode = lr["mode"]
    camera_axis_offset_deg = None
    camera_ring_radius_m = None
    camera_depth_fraction = None

    if placement_mode == "camera_ring" and cam is not None:
        axis = _norm(np.asarray(cam["origin"], float) - tgt)  # từ vật thể hướng về camera
        camera_dist = float(np.linalg.norm(np.asarray(cam["origin"], float) - tgt))
        if camera_dist <= 1e-8:
            raise ValueError("Camera target distance quá nhỏ để đặt đèn CLEAN")

        # Mặt phẳng vòng đèn nằm ở phía trước camera một đoạn, tránh z_camera=0.
        # Góc ngoài khung = nửa FOV ngang + margin; margin còn chừa chỗ cho kích thước panel.
        camera_depth_fraction = float(rng.uniform(*lr["depth_fraction"]))
        margin_deg = float(rng.uniform(*lr["frustum_margin_deg"]))
        camera_axis_offset_deg = min(70.0, float(cam["fov_deg"]) / 2.0 + margin_deg)
        camera_ring_radius_m = (camera_depth_fraction * camera_dist
                                * math.tan(math.radians(camera_axis_offset_deg)))

        # Tạo hai vector vuông góc với trục nhìn, rồi lấy vị trí ngẫu nhiên trên đường tròn.
        ref = np.array([0.0, 1.0, 0.0]) if abs(axis[1]) < 0.90 else np.array([1.0, 0.0, 0.0])
        b1 = _norm(np.cross(axis, ref))
        b2 = _norm(np.cross(axis, b1))
        phi = math.radians(float(rng.uniform(0.0, 360.0)))
        lateral = math.cos(phi) * b1 + math.sin(phi) * b2
        cam_origin = np.asarray(cam["origin"], float)
        toward_target = (tgt - cam_origin)
        pos = cam_origin + camera_depth_fraction * toward_target + camera_ring_radius_m * lateral
        dist = float(np.linalg.norm(pos - tgt))
        direction = _norm(pos - tgt)
        el = math.degrees(math.asin(float(np.clip(direction[2], -1.0, 1.0))))
        az = math.degrees(math.atan2(direction[1], direction[0])) % 360.0
    else:
        el = float(rng.uniform(*lr["elev_deg"]))
        az = float(rng.uniform(0.0, 360.0))
        dist = float(rng.uniform(*cfg.light_dist_m) * scale)
        direction = np.array([
            math.cos(math.radians(el)) * math.cos(math.radians(az)),
            math.cos(math.radians(el)) * math.sin(math.radians(az)),
            math.sin(math.radians(el)),
        ], dtype=float)
        pos = tgt + dist * direction

    # Giữ mức chiếu sáng phụ thuộc khoảng cách/kích thước như bản gốc, thêm hệ số profile.
    L = (dist ** 2 / max(size ** 2, 1e-8)) * power_mult
    warm = rng.uniform(0.90, 1.0)
    items = {"key": rect_emitter(pos, tgt, size, [L, L * (0.9 + 0.1 * warm), L * warm])}
    fill = float(rng.uniform(*lr["fill_ratio"]))
    items["env"] = {"type": "constant", "radiance": {"type": "rgb", "value": fill / math.pi}}
    info = {
        "placement_mode": placement_mode,
        "key_elev_deg": float(el),
        "key_azim_deg": float(az),
        "key_dist_m": float(dist),
        "key_size_m": float(size),
        "key_pos": np.asarray(pos, float).tolist(),
        "fill_ratio": fill,
        "key_power_multiplier": power_mult,
        "second_light": False,
    }
    if camera_axis_offset_deg is not None:
        info.update({
            "camera_axis_offset_deg": float(camera_axis_offset_deg),
            "camera_ring_radius_m": float(camera_ring_radius_m),
            "camera_depth_fraction": float(camera_depth_fraction),
            "frustum_margin_deg": float(margin_deg),
        })

    second_prob = float(lr["second_light_prob"])
    if rng.random() < second_prob:
        el2 = rng.uniform(20, 80)
        az2 = rng.uniform(0, 360)
        d2 = rng.uniform(0.6, 2.0) * scale
        s2 = rng.uniform(0.1, 0.6) * scale
        k2 = rng.uniform(0.15, 0.5) * power_mult
        p2 = tgt + d2 * np.array([
            math.cos(math.radians(el2)) * math.cos(math.radians(az2)),
            math.cos(math.radians(el2)) * math.sin(math.radians(az2)),
            math.sin(math.radians(el2)),
        ])
        L2 = k2 * d2 ** 2 / max(s2 ** 2, 1e-8)
        items["key2"] = rect_emitter(p2, tgt, s2, [L2, L2, L2])
        info.update({"second_light": True, "l2_elev_deg": el2, "l2_azim_deg": az2,
                     "l2_strength": k2, "l2_pos": p2.tolist(), "l2_size_m": s2})
    return items, info

def light_in_frame(cam, pos, size, margin_px=4.0):
    """Tấm đèn (hình chữ nhật, bán kính bao ~0.71*cạnh) có lọt vào khung hình camera không?
    Mặt sau tấm đèn không phát sáng nên hiện thành mảng ĐEN che ảnh (đã gặp trên ảnh thật)."""
    r = 0.71 * size
    uv, z, _ = project(cam, np.asarray(pos, float)[None])
    z = float(z[0])
    if z <= -r:
        return False                      # hoàn toàn sau lưng camera
    if z <= r:
        return True                       # sát mặt phẳng camera -> chắn tầm nhìn
    rp = cam["K"][0, 0] * r / z + margin_px
    u, v = uv[0]
    return bool(-rp < u < cam["W"] + rp and -rp < v < cam["H"] + rp)

def sample_lights_outside(rng, cam, scale=1.0, data_profile="normal", detail_level="sharp"):
    """Bốc đèn theo profile cho tới khi không tấm đèn nào lọt vào khung hình."""
    for n in range(cfg.light_resample_max):
        items, info = sample_lights(rng, cam["target"], scale, data_profile=data_profile, cam=cam, detail_level=detail_level)
        bad = light_in_frame(cam, info["key_pos"], info["key_size_m"]) or (
            info["second_light"] and light_in_frame(cam, info["l2_pos"], info["l2_size_m"]))
        if not bad:
            info["light_resamples"] = n
            return items, info
    return None, None                     # góc camera này quá chật: render_one sẽ bốc góc khác

BACKGROUNDS = {"white": (0.80, 0.80, 0.78), "light_gray": (0.50, 0.50, 0.50),
               "dark_cloth": (0.05, 0.05, 0.06), "wood_table": (0.30, 0.19, 0.11)}

def build_scene(cam, ply_path, tex_path, rough, bg_name, lights, spp, zmin, gs = 4.0):
    bg = BACKGROUNDS[bg_name]
    sc = {"type": "scene", "integrator": {"type": "path", "max_depth": 6},
          "sensor": mi_sensor(cam, spp),
          "obj": {"type": "ply", "filename": str(ply_path),
                  "bsdf": {"type": "principled",
                           "base_color": {"type": "bitmap", "filename": str(tex_path)},
                           "roughness": rough, "specular": 0.4}},
          "ground": {"type": "rectangle", "to_world": T.translate([0, 0, zmin - 2e-4]) @ T.scale([gs, gs, 1]),
                     "bsdf": {"type": "diffuse", "reflectance": {"type": "rgb", "value": list(bg)}}}}
    sc.update(lights)
    return mi.load_dict(sc)

def render_albedo(cam, ply_path, tex_path, zmin, bg_rgb, spp=4):
    """Ảnh MÀU VẬT LIỆU (albedo) của cùng khung hình: màu gỗ và mực, không có ánh sáng.
    Ảnh render chia cho ảnh này = bóng đổ thuần, dùng để đo chữ có hiện lên không."""
    sc = mi.load_dict({"type": "scene", "integrator": {"type": "aov", "aovs": "alb:albedo"},
                       "sensor": mi_sensor(cam, spp),
                       "obj": {"type": "ply", "filename": str(ply_path),
                               "bsdf": {"type": "principled",
                                        "base_color": {"type": "bitmap", "filename": str(tex_path)}}},
                       "ground": {"type": "rectangle",
                                  "to_world": T.translate([0, 0, zmin - 2e-4]) @ T.scale([4, 4, 1]),
                                  "bsdf": {"type": "diffuse", "reflectance": {"type": "rgb", "value": list(bg_rgb)}}}})
    return np.array(mi.render(sc, spp=spp))[..., :3]

def render_depth(cam, ply_path):
    """Tia 1 mẫu/pixel, chỉ có vật thể -> mặt nạ vật thể (depth > 0).
    Lưu ý: giá trị depth của Mitsuba tính từ mặt phẳng near_clip, KHÔNG dùng để so khoảng cách."""
    sc = mi.load_dict({"type": "scene", "integrator": {"type": "aov", "aovs": "dd:depth"},
                       "sensor": mi_sensor(cam, 1, rfilter="box"),
                       "obj": {"type": "ply", "filename": str(ply_path)}})
    a = np.array(mi.render(sc, spp=1))
    depth = a[..., -1]
    return depth, depth > 0

## 9. Hàm kiểm tra phép chiếu
Đặt 8 quả cầu phát sáng tại điểm 3D đã biết, render, dò tâm trên ảnh, so với phép chiếu numpy. Bắt được lỗi lật trục x/y, lệch FOV, lệch nửa pixel. Ảnh kiểm tra lưu ở `output/selftest/` (vòng đỏ = vị trí dự đoán).

In [ ]:
def projection_selftest(mesh, n_cams=3, W=640, H=480, spp=8, seed=123):
    """Đặt 8 quả cầu phát sáng tại điểm 3D đã biết, render, dò tâm trên ảnh,
    so với phép chiếu numpy. Bắt được lỗi lật trục x/y và lệch FOV."""
    rng = np.random.default_rng(seed)
    lo, hi = mesh.bounds
    c = (lo + hi) / 2; e = (hi - lo) / 2
    rad = 0.012 * np.linalg.norm(hi - lo)
    offs = np.array([[-.7, -.6, 0], [.7, -.6, 0], [.7, .6, 0], [-.7, .6, 0],
                     [0, 0, .5], [.3, -.2, .9], [-.4, .3, .9], [.1, .45, .2]])
    pts = c + offs * np.r_[e[:2], max(e[2], 0.15 * e[:2].max())]
    errs, report = [], []
    for k in range(n_cams):
        cam, _ = sample_camera(mesh, rng, W, H, plate_like=False)
        sc = {"type": "scene", "integrator": {"type": "path", "max_depth": 2}, "sensor": mi_sensor(cam, spp)}
        for i, p in enumerate(pts):
            sc[f"s{i}"] = {"type": "sphere", "center": p.tolist(), "radius": float(rad),
                           "emitter": {"type": "area", "radiance": {"type": "rgb", "value": 1.0}}}
        img = np.array(mi.render(mi.load_dict(sc), spp=spp)).mean(-1)
        n, lab, stats, cents = cv2.connectedComponentsWithStats((img > 0.3 * img.max()).astype(np.uint8))
        found = cents[1:][stats[1:, cv2.CC_STAT_AREA] >= 3] + 0.5   # OpenCV: tâm pixel ở số nguyên -> +0.5
        uv, z, _ = project(cam, pts)
        vis = (z > 0) & (uv[:, 0] > 5) & (uv[:, 0] < W - 5) & (uv[:, 1] > 5) & (uv[:, 1] < H - 5)
        e_cam = []
        for p in uv[vis]:
            if len(found):
                e_cam.append(np.min(np.linalg.norm(found - p, axis=1)))
        errs += e_cam
        # lưu ảnh để nhìn bằng mắt: vòng đỏ = dự đoán numpy
        viz = cv2.cvtColor((np.clip(img / img.max(), 0, 1) * 255).astype(np.uint8), cv2.COLOR_GRAY2BGR)
        for p in uv[vis]:
            cv2.circle(viz, (int(round(p[0] - 0.5)), int(round(p[1] - 0.5))), 9, (0, 0, 255), 1)
        cv2.imwrite(str(ROOT / "selftest" / f"projection_cam{k}.png"), viz)
        report.append({"cam": k, "visible_points": int(vis.sum()), "detected": int(len(found)),
                       "mean_err_px": float(np.mean(e_cam)) if e_cam else None})
    mean_err = float(np.mean(errs)) if errs else float("inf")
    return mean_err, report

## 10. Chuẩn bị vật thể
Nạp mesh theo nguồn đã chọn, chuẩn hóa tư thế, ghi PLY có UV. Nếu vật thể dạng tấm mà không có nhãn, sinh **lưới nhãn giả** (`pseudo: true`) chỉ để kiểm tra phép chiếu.

In [ ]:
import zlib

def prepare_mesh_items():
    """Trả về danh sách vật thể cần render: (tên, mesh gốc, nhãn ký tự, nhãn cột, có_chữ_thật)."""
    items = []
    if cfg.mesh_source == "synthetic":
        m, ch, co = make_synthetic_block()
        items.append(("synthetic", m, ch, co, True))
    elif cfg.mesh_source == "hf":
        from huggingface_hub import hf_hub_download
        names = cfg.hf_filenames if getattr(cfg, "hf_filenames", None) else [cfg.hf_filename]
        for nm in names:
            try:
                p = hf_hub_download(repo_id=cfg.hf_repo_id, filename=nm, repo_type="dataset")
                items.append((Path(nm).parts[0], load_mesh_any(p), [], [], False))
            except Exception as e:
                print(f"[WARN] tải {nm} thất bại: {e}")
                FALLBACKS.append(f"tải HF {nm} thất bại -> dùng ván giả")
        if not items:
            m, ch, co = make_synthetic_block()
            items.append(("synthetic", m, ch, co, True))
    elif cfg.mesh_source == "file":
        paths = cfg.mesh_paths or ([cfg.mesh_path] if cfg.mesh_path else [])
        if not paths:
            raise RuntimeError("Không có mesh nào. Kiểm tra đã Add Input dataset chứa file .stl chưa")
        for mp in paths:
            m = load_mesh_any(mp)
            # nhãn ngoài chỉ áp dụng khi có đúng một file
            label_path = cfg.labels_path if len(paths)==1 else ""
            if cfg.annotation_index_path:
                ip=Path(cfg.annotation_index_path)
                entry=json.loads(ip.read_text(encoding="utf-8")).get("boards",{}).get(Path(mp).stem,{})
                if entry.get("labels_path"):
                    label_path=str((ip.parent/entry["labels_path"]).resolve())
            ch, co = load_external_labels(label_path) if label_path else ([], [])
            items.append((Path(mp).stem, m, ch, co, bool(ch)))
    else:
        raise ValueError(cfg.mesh_source)
    return items

def setup_item(tag, mesh, chars, cols, has_text):
    """Chuẩn hóa tư thế (cùng một ma trận cho mesh và nhãn), ghi PLY có UV, chuẩn bị mặt nạ mực."""
    m2, M, plate, info = normalize_pose(mesh)

    if plate and cfg.board_rot_z_deg:
        Rz = trimesh.transformations.rotation_matrix(math.radians(cfg.board_rot_z_deg), [0, 0, 1])
        m2.apply_transform(Rz); M = Rz @ M          # M đổi thì nhãn đổi theo
    chars, cols = apply_M(chars, M), apply_M(cols, M)   # dòng cũ, giữ nguyên
    

    if plate and not chars:
        chars = pseudo_grid_labels(m2)
        print(f"[{tag}] dạng tấm nhưng không có nhãn -> dùng lưới giả {cfg.pseudo_rows}x{cfg.pseudo_cols}")
    ply = ROOT / "assets" / f"{tag}.ply"
    write_ply_uv(m2, ply)
    probe = mi.load_dict({"type": "ply", "filename": str(ply)})
    assert probe.has_vertex_texcoords(), "PLY không có UV — vật liệu gỗ sẽ không bám bề mặt"
    raised = raised_mask_uv(m2, ply) if plate else None
    print(f"[{tag}] plate_like={plate} (tỉ lệ trị riêng {info['eig_ratio']:.4f}) | "
          f"kích thước sau chuẩn hóa (mm) = {np.round(m2.extents * 1000, 1)} | "
          f"{len(chars)} nhãn chữ, {len(cols)} nhãn cột")
    item = {"tag": tag, "mesh": m2, "M": M, "plate": plate, "pose_info": info, "chars": chars,
            "cols": cols, "has_text": has_text, "ply": ply, "raised": raised,
            "zmin": float(m2.bounds[0][2])}
    item['source_mesh_path'] = str(next((p for p in cfg.mesh_paths if Path(p).stem==tag), 'synthetic'))
    sm = globals().get('SYNTHETIC_MASKS') if cfg.mesh_source=='synthetic' else None
    item = prepare_research_item(item,cfg,ROOT,mi,occlusion_scene(ply),sm)
    item = attach_visibility_points(item)
    return item


## 11. Phơi sáng và hiệu ứng máy ảnh
Render ra ảnh HDR, rồi **tự phơi sáng**: đưa trung vị độ sáng vật thể về mục tiêu ngẫu nhiên, chặn trên để không cháy sáng (khắc phục lỗi cháy sáng của lần thử bằng Blender). Sau đó: lệch cân bằng trắng, vignette, mất nét, rung tay (xác suất 0.2), nhiễu cảm biến, nén JPEG.

In [ ]:

def srgb(x):
    x = np.clip(x, 0, 1)
    return np.where(x <= 0.0031308, 12.92 * x, 1.055 * np.power(x, 1 / 2.4) - 0.055)

def srgb_inv(y):
    return np.where(y <= 0.04045, y / 12.92, ((y + 0.055) / 1.055) ** 2.4)

def luminance(img):
    return img[..., 0] * 0.2126 + img[..., 1] * 0.7152 + img[..., 2] * 0.0722

# OCR post-processing: all camera effects are controlled by data_profile.
def enhance_for_ocr(img, rng, data_profile="clean", detail_level="sharp"):
    """Sharpen nhẹ trên kênh L; thông số phụ thuộc profile OCR."""
    skip_prob = float(get_profile_value(data_profile, "sharpen_skip_prob", detail_level=detail_level))
    if (not cfg.sharpen_enabled) or rng.random() < skip_prob:
        return img, {"skipped": True, "skip_prob": skip_prob}

    amt = rng.uniform(*get_profile_value(data_profile, "sharpen_amount", detail_level=detail_level))
    sig = rng.uniform(*get_profile_value(data_profile, "sharpen_sigma_px", detail_level=detail_level))
    lab = cv2.cvtColor(np.clip(img, 0, 1).astype(np.float32), cv2.COLOR_RGB2LAB)
    L = lab[..., 0]  # float32: L trong 0..100
    lab[..., 0] = np.clip(
        L + amt * (L - cv2.GaussianBlur(L, (0, 0), sig)),
        0, 100,
    )
    out = np.clip(cv2.cvtColor(lab, cv2.COLOR_LAB2RGB), 0, 1)
    return out, {
        "skipped": False,
        "unsharp_amount": float(amt),
        "unsharp_sigma_px": float(sig),
        "skip_prob": skip_prob,
    }


def camera_effects(img, rng, enh_rng=None, data_profile="clean", detail_level="sharp"):
    """
    Hiệu ứng camera OCR theo profile + detail level:
      clean  : tương đương profile normal cũ (camera gần cfg mặc định, sharpen rõ hơn)
      normal : tương đương profile clean cũ (hiệu ứng camera nhẹ, sharpen bảo thủ hơn)
      hard   : blur/noise/JPEG suy giảm mạnh hơn để tăng độ khó
    img là ảnh sRGB float trong [0, 1].
    """
    if data_profile not in PROFILE_WEIGHTS:
        raise ValueError(f"Unknown data profile: {data_profile}")

    H, W = img.shape[:2]
    info = {"data_profile": data_profile, "detail_level": detail_level}

    # White balance và vignette được áp dụng trong không gian tuyến tính.
    lin = srgb_inv(np.clip(img, 0, 1)).astype(np.float32)
    wb_range = get_profile_value(data_profile, "wb_gain", detail_level=detail_level)
    g = rng.uniform(*wb_range, 3)
    lin = lin * g
    info["wb_gain"] = g.tolist()

    v = float(rng.uniform(*get_profile_value(data_profile, "vignette", detail_level=detail_level)))
    yy, xx = np.mgrid[0:H, 0:W]
    r2 = ((xx - W / 2) / (W / 2)) ** 2 + ((yy - H / 2) / (H / 2)) ** 2
    lin = lin * (1 - v * np.clip(r2 / 2, 0, 1))[..., None]
    info["vignette"] = v
    img = srgb(np.clip(lin, 0, 1)).astype(np.float32)

    # Defocus blur.
    s = float(rng.uniform(*get_profile_value(data_profile, "defocus_sigma_px", detail_level=detail_level)))
    if s > 0.2:
        img = cv2.GaussianBlur(img, (0, 0), s)
    info["defocus_sigma_px"] = s

    # Motion blur.
    motion_prob = float(get_profile_value(data_profile, "motion_blur_prob", detail_level=detail_level))
    if rng.random() < motion_prob:
        L = int(rng.integers(2, 5))
        ang = float(rng.uniform(0, 180))
        k = np.zeros((L, L), np.float32)
        k[L // 2, :] = 1
        k = cv2.warpAffine(
            k,
            cv2.getRotationMatrix2D((L / 2 - 0.5, L / 2 - 0.5), ang, 1),
            (L, L),
        )
        img = cv2.filter2D(img, -1, k / max(float(k.sum()), 1e-6))
        info["motion_blur"] = {"len_px": L, "angle_deg": ang}
    else:
        info["motion_blur"] = None
    info["motion_blur_prob"] = motion_prob

    # Shot/read noise.
    a = float(rng.uniform(*get_profile_value(data_profile, "noise_a", detail_level=detail_level)))
    b = float(rng.uniform(*get_profile_value(data_profile, "noise_b", detail_level=detail_level)))
    img = img + rng.normal(0, 1, img.shape) * np.sqrt(
        np.clip(a * img + b, 0, None)
    )
    info["noise"] = {"a": a, "b": b}

    # Sharpen profile được áp dụng sau blur/noise và trước JPEG.
    enhancer_rng = enh_rng if enh_rng is not None else np.random.default_rng(0)
    img, info["enhance"] = enhance_for_ocr(img, enhancer_rng, data_profile, detail_level=detail_level)

    # JPEG compression.
    qmin, qmax = get_profile_value(data_profile, "jpeg_quality", detail_level=detail_level)
    q = int(rng.integers(int(qmin), int(qmax) + 1))
    info["jpeg_quality"] = q
    u8 = (np.clip(img, 0, 1) * 255 + 0.5).astype(np.uint8)
    ok, buf = cv2.imencode(
        ".jpg",
        cv2.cvtColor(u8, cv2.COLOR_RGB2BGR),
        [cv2.IMWRITE_JPEG_QUALITY, q],
    )
    if not ok:
        raise RuntimeError("cv2.imencode thất bại khi tạo JPEG")
    return buf, info


## 12. Chiếu nhãn và kiểm tra che khuất + thêm texture + hiệu ứng bề mặt

Chiếu 4 góc mỗi nhãn bằng đúng K, R, t của camera. Nhãn **nhìn thấy** khi cả 4 góc nằm trong khung và không bị vật thể che (bắn tia từ tâm camera tới từng góc).

# 12.1 : CHIẾU NHÃN VÀ CHE KHUẤT

In [ ]:
def ray_hit_dist(scene, origin, pts):
    """Bắn tia từ ĐÚNG tâm camera tới từng điểm, trả về khoảng cách tới mặt đầu tiên bị chạm.
    (Không dùng AOV depth: Mitsuba đo nó từ mặt phẳng near_clip, không phải từ tâm camera.)"""
    pts = np.asarray(pts, float); o = np.asarray(origin, float)
    d = pts - o; d /= np.linalg.norm(d, axis=1, keepdims=True)
    if VARIANT == "scalar_rgb":
        out = []
        for di in d:
            si = scene.ray_intersect(mi.Ray3f(mi.Point3f(*o), mi.Vector3f(*di)))
            out.append(float(si.t))
        return np.array(out)
    n = len(d)
    ray = mi.Ray3f(mi.Point3f(np.full(n, o[0]), np.full(n, o[1]), np.full(n, o[2])),
                   mi.Vector3f(d[:, 0].copy(), d[:, 1].copy(), d[:, 2].copy()))
    return np.array(scene.ray_intersect(ray).t)

def project_labels(items, cam, occ_scene):
    if not items:
        return []
    W, H = cam["W"], cam["H"]
    Q = np.concatenate([np.asarray(it["quad"], float) for it in items])        # (4N, 3)
    uv, z, dist = project(cam, Q)
    t = ray_hit_dist(occ_scene, cam["origin"], Q)
    inside = (z > 0) & (uv[:, 0] >= 0) & (uv[:, 0] <= W) & (uv[:, 1] >= 0) & (uv[:, 1] <= H)
    unocc = ~(t < dist - cfg.visibility_tol_m)          # tia chạm vật thể TRƯỚC điểm nhãn -> bị che
    ok = (inside & unocc).reshape(-1, 4).all(1)
    out = []
    for k, it in enumerate(items):
        rec = {kk: it[kk] for kk in ("char", "text", "col", "row", "pseudo") if kk in it}
        rec["poly_px"] = np.round(uv[4 * k:4 * k + 4], 2).tolist()
        rec["visible"] = bool(ok[k])
        out.append(rec)
    return out

def poly_mask(poly, H, W):
    m = np.zeros((H, W), np.uint8)
    cv2.fillPoly(m, [np.round(np.asarray(poly) - 0.5).astype(np.int32)], 1)
    return m.astype(bool)

def shading_contrast(hdr, albedo, mask, chars_px):
    """Tương phản của ảnh BÓNG ĐỔ THUẦN (render / albedo) trong các ô nhãn nhìn thấy.
    Chia cho albedo triệt tiêu vân gỗ và vết mực, chỉ còn hình khối hiện lên nhờ ánh sáng —
    nên đo đúng chữ có đọc được hay đã bị đèn chiếu thẳng xóa mất.
    (Bản trước đo trên ảnh màu nên bị vân gỗ đánh lừa: ảnh mất chữ vẫn đạt 0.109.)"""
    Ls = luminance(hdr) / np.maximum(luminance(albedo), 1e-4)
    H, W = Ls.shape; vals = []
    for c in chars_px:
        if c["visible"]:
            m = poly_mask(c["poly_px"], H, W) & mask
            if m.sum() > 50:
                v = Ls[m]
                lo, hi = np.percentile(v, [2, 98])          # bỏ điểm cực trị ở mép bóng
                v = v[(v >= lo) & (v <= hi)]
                vals.append(v.std() / max(v.mean(), 1e-6))
    return float(np.median(vals)) if vals else None



# 12.2 : LOAD TEXTURE BỀ MẶT GỖ : SỬ DỤNG OAK WOOD

In [ ]:
WOOD_NAME = "Natural-Oak-Wood-Texture.jpg"
_hits = sorted(Path(cfg.texture_dir).rglob(WOOD_NAME)) if Path(cfg.texture_dir).exists() else []
cfg.texture_paths = [str(_hits[0])] if _hits else []
_WOOD_IMG = None
if cfg.texture_paths:
    _bgr = cv2.imread(cfg.texture_paths[0],cv2.IMREAD_COLOR)
    if _bgr is None:raise ValueError("Unreadable wood texture: "+cfg.texture_paths[0])
    _WOOD_IMG = cv2.cvtColor(_bgr,cv2.COLOR_BGR2RGB)
else:
    print("No external wood texture; procedural material is recorded explicitly.")

def load_ext_texture(rng,S=1024):
    if _WOOD_IMG is None:return make_wood_texture(rng,S)
    img=_WOOD_IMG;h,w=img.shape[:2];s=int(min(h,w)*rng.uniform(.6,1.0))
    y0=int(rng.integers(0,h-s+1));x0=int(rng.integers(0,w-s+1))
    img=np.rot90(img[y0:y0+s,x0:x0+s],int(rng.integers(0,4)))
    if rng.random()<.5:img=img[:,::-1]
    tex=cv2.resize(np.ascontiguousarray(img),(S,S),interpolation=cv2.INTER_AREA)
    return srgb_inv(tex.astype(np.float32)/255).astype(np.float32),{
        "texture":"external","file":cfg.texture_paths[0],"crop_xywh":[x0,y0,s,s]}


# 12.3 : TẠO HIỆU ỨNG XƯỚC BỀ MẶT / BỤI / NỨT

In [ ]:
# =====================================================================
# 7b. WEAR LAYER: cracks, wear marks, dust/grime, age grade  (albedo / texture only)
# Paste as a NEW code cell right before "13. Render một ảnh"
# (after the cell that defines load_ext_texture, so srgb_inv, fractal_noise,
#  occlusion_scene, mi, cv2 and `dataclass` already exist).
# Nothing here touches lights, exposure, sharpening, camera effects or Config.
# =====================================================================
from dataclasses import dataclass
import math, time
import numpy as np
import cv2


@dataclass
class WearConfig:
    enabled: bool = True                      # master switch: False -> pipeline behaves exactly as before

    # --- 1. cracks (checks along the grain) ---
    n_edge_cracks: tuple = (2, 6)             # start near the outline, mostly on end-grain edges
    n_floor_cracks: tuple = (3, 8)            # fine hairlines in the open floor
    crack_len_mm: tuple = (10.0, 70.0)
    crack_width_mm: tuple = (0.25, 0.55)      # ~1 render pixel is 0.3-0.4 mm: thinner = faint grey line
    crack_dark: tuple = (0.55, 0.80)          # max darkening inside a crack
    hairline_len_mm: tuple = (6.0, 30.0)
    hairline_width_mm: tuple = (0.15, 0.30)
    hairline_dark: tuple = (0.25, 0.45)
    crack_wander_deg: float = 4.0
    crack_branch_prob: float = 0.03
    min_grain_coherence: float = 0.12         # below this the grain estimate is unreliable -> use plate long axis

    # --- 2. wear marks ---
    scratch_per_cm2: tuple = (0.04, 0.12)
    scratch_len_mm: tuple = (3.0, 25.0)
    scratch_width_mm: tuple = (0.12, 0.25)
    scratch_strength: tuple = (0.08, 0.22)
    scratch_light_frac: float = 0.5           # share of scratches that are light (fresh wood) vs dark (dirt)
    scratch_angle_jitter_deg: float = 12.0
    edge_wear_band_mm: float = 2.0            # handled-edge darkening, e-folding distance from the outline
    edge_wear_strength: tuple = (0.06, 0.16)

    # --- 3. dust and grime ---
    grime_strength: tuple = (0.10, 0.22)      # max blend toward grime colour in the deepest recesses
    grime_gamma: float = 0.8
    dust_strength: tuple = (0.10, 0.30)       # pale film on the floor, patchy
    speck_per_cm2: tuple = (0.3, 1.2)
    speck_diam_mm: tuple = (0.4, 1.1)         # smaller than this vanishes at ~0.35 mm/pixel
    speck_strength: tuple = (0.30, 0.60)

    # --- 5. very light dust on letter tops (can only lighten/desaturate, never darken) ---
    letter_dust: bool = True
    letter_dust_strength: tuple = (0.03, 0.05)

    # --- 4. age grade: colour/tone of the wood itself (desaturate, grey-brown shift, soft mottling) ---
    age_grade: bool = True                    # own switch: False -> no colour grade, everything else unchanged
    age_desat: tuple = (0.15, 0.30)           # fraction of colour saturation removed (luminance kept)
    age_tint: tuple = (0.10, 0.20)            # blend toward a grey-brown reference (luminance kept)
    age_mottle: tuple = (0.06, 0.10)          # peak +/- low-frequency tonal variation (typical swing ~40% of this)
    age_on_letters: bool = True               # True: whole board is graded, so letters keep matching the floor

    # --- guard rails ---
    guard_mm: float = 0.3                     # exclusion margin around the raised letters
    edge_fade_mm: tuple = (0.8, 2.5)          # cracks/scratches fade to 0 at the outline (planar UV smears the sides)
    cavity_sigma_mm: float = 2.0              # neighbourhood used to decide what counts as a recess


WEAR = WearConfig()

_LUM = np.array([0.2126, 0.7152, 0.0722], np.float32)
_TINT = np.array([1.0, 1.06, 1.18], np.float32)                       # dirt darkens blue a bit more -> brownish
_GRIME = np.asarray(srgb_inv(np.array([0.26, 0.22, 0.18])), np.float32)
_AGE_REF = np.asarray(srgb_inv(np.array([0.45, 0.39, 0.33])), np.float32)   # grey-brown reference colour
_AGE_DIR = _AGE_REF / float(_AGE_REF @ _LUM)                                  # same hue, unit luminance
_WEAR_CACHE = {}


def _smooth(x, a, b):
    t = np.clip((x - a) / max(b - a, 1e-9), 0.0, 1.0)
    return t * t * (3.0 - 2.0 * t)


def _patchy(rng, shape, octaves=5, size=512):
    """Low-frequency noise in [0,1] stretched to the texture grid."""
    n = fractal_noise(rng, size, octaves=octaves)
    return cv2.resize(n, (shape[1], shape[0]), interpolation=cv2.INTER_CUBIC)


# ---------------------------------------------------------------------
# Per-board maps (computed ONCE per board and cached): footprint, distance to the
# outline, "cavity" (how far a point sits below its surroundings), edge seed points.
# Same top-down ray grid as raised_mask_uv, so everything is pixel-aligned with the texture.
# ---------------------------------------------------------------------
def _height_grid(item, shape):
    Hm, Wm = shape
    lo, hi = item["mesh"].bounds
    ext = hi - lo
    xs = lo[0] + (np.arange(Wm) + 0.5) / Wm * ext[0]
    ys = hi[1] - (np.arange(Hm) + 0.5) / Hm * ext[1]                  # row 0 = largest y (same as raised_mask_uv)
    sc = occlusion_scene(item["ply"])
    z = np.full((Hm, Wm), np.nan, np.float32)
    step = max(1, 2_000_000 // Wm)
    for r0 in range(0, Hm, step):
        r1 = min(Hm, r0 + step)
        X, Y = np.meshgrid(xs, ys[r0:r1]); n = X.size
        ray = mi.Ray3f(mi.Point3f(X.ravel().copy(), Y.ravel().copy(), np.full(n, hi[2] + 0.05)),
                       mi.Vector3f(np.zeros(n), np.zeros(n), -np.ones(n)))
        t = np.array(sc.ray_intersect(ray).t).reshape(r1 - r0, Wm)
        z[r0:r1] = np.where(np.isfinite(t), hi[2] + 0.05 - t, np.nan)
    return z


def board_maps(item, shape, wc):
    key = (item["tag"], item.get("mesh_sha256"), tuple(shape), wc.cavity_sigma_mm)
    if key in _WEAR_CACHE:
        return _WEAR_CACHE[key]
    t0 = time.time()
    Hm, Wm = shape
    lo, hi = item["mesh"].bounds
    texel = float((hi[0] - lo[0]) * 1000.0 / Wm)                        # mm per texel
    z = _height_grid(item, shape)
    foot = np.isfinite(z)
    zf = np.where(foot, z, np.nanmedian(z)).astype(np.float32)
    blur = cv2.GaussianBlur(zf, (0, 0), wc.cavity_sigma_mm / texel)
    cav = np.clip(blur - zf, 0.0, None)
    ref = float(np.percentile(cav[foot][::37], 95)) if foot.any() else 1.0
    cav = (np.clip(cav / max(ref, 1e-9), 0.0, 1.0) * foot).astype(np.float32)
    dist_mm = (cv2.distanceTransform(foot.astype(np.uint8), cv2.DIST_L2, 3) * texel).astype(np.float32)
    # seed points for edge cracks: coarse grid, 1.5-6 mm inside the outline, with the inward normal
    s = 4
    ds = cv2.resize(dist_mm, (max(8, Wm // s), max(8, Hm // s)), interpolation=cv2.INTER_AREA)
    gy, gx = np.gradient(cv2.GaussianBlur(ds, (0, 0), 2.0))             # gradient of distance = inward normal
    nrm = np.hypot(gx, gy) + 1e-9
    yy, xx = np.nonzero((ds > 1.5) & (ds < 6.0))
    cand = np.stack([xx * s + s / 2, yy * s + s / 2, gx[yy, xx] / nrm[yy, xx], gy[yy, xx] / nrm[yy, xx]], 1).astype(np.float32)
    out = {"texel_mm": texel, "foot": foot, "dist_mm": dist_mm, "cav": cav, "edge_cand": cand}
    _WEAR_CACHE[key] = out
    area = foot.sum() * texel * texel / 100.0
    print(f"[wear] board maps for {item['tag']}: grid {Wm}x{Hm}, {texel:.3f} mm/texel, "
          f"footprint {area:.0f} cm2, cavity p95 {ref * 1000:.3f} mm, {len(cand):,} edge seeds ({time.time() - t0:.1f}s)")
    return out


# ---------------------------------------------------------------------
# Grain direction, estimated from the texture itself (structure tensor).
# Needed because load_ext_texture rotates the photo by random multiples of 90 deg.
# ---------------------------------------------------------------------
def grain_angle(tex):
    h, w = tex.shape[:2]
    s = 1024.0 / max(h, w)
    small = cv2.resize(tex, (max(8, int(w * s)), max(8, int(h * s))), interpolation=cv2.INTER_AREA)
    g = cv2.GaussianBlur(small @ _LUM, (0, 0), 1.5)
    gx = cv2.Sobel(g, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(g, cv2.CV_32F, 0, 1, ksize=3)
    jxx, jyy, jxy = float((gx * gx).mean()), float((gy * gy).mean()), float((gx * gy).mean())
    coh = math.hypot(jxx - jyy, 2 * jxy) / (jxx + jyy + 1e-12)
    theta = 0.5 * math.atan2(2 * jxy, jxx - jyy) + math.pi / 2         # fibres run perpendicular to the gradient
    return theta, coh


# ---------------------------------------------------------------------
# Drawing helpers (anti-aliased lines on uint8 canvases; value = relative intensity)
# ---------------------------------------------------------------------
def _walk(canvas, x, y, ang0, length_mm, width_mm, val, rng, texel, wander_deg, branch_p, depth=0):
    """One crack: a correlated random walk that is pulled back toward its start direction (the grain),
    tapering to a point, with rare thinner side branches."""
    step_mm = 0.6
    step = step_mm / texel
    n = max(2, int(length_mm / step_mm))
    H, Wc = canvas.shape
    ang, px, py = ang0, x, y
    for k in range(n):
        s = k / (n - 1)
        d = (ang0 - ang + math.pi) % (2 * math.pi) - math.pi
        ang += 0.2 * d + rng.normal(0.0, math.radians(wander_deg))
        qx, qy = px + step * math.cos(ang), py + step * math.sin(ang)
        if not (0 <= qx < Wc and 0 <= qy < H):
            break
        w_px = max(1.0, width_mm / texel * (1.0 - s) ** 0.8)
        cv2.line(canvas, (int(round(px)), int(round(py))), (int(round(qx)), int(round(qy))),
                 int(val), int(round(w_px)), cv2.LINE_AA)
        if depth == 0 and 0.15 < s < 0.85 and rng.random() < branch_p:
            sgn = 1.0 if rng.random() < 0.5 else -1.0
            _walk(canvas, qx, qy, ang + sgn * math.radians(rng.uniform(15, 35)),
                  length_mm * (1 - s) * rng.uniform(0.25, 0.5), width_mm * 0.5 * (1 - s), val,
                  rng, texel, wander_deg, 0.0, depth=1)
        px, py = qx, qy


def _scratch(canvas, x, y, ang, length_mm, width_mm, val, rng, texel):
    n, seg = 3, length_mm / texel / 3
    th = max(1, int(round(width_mm / texel)))
    px, py = x, y
    for _ in range(n):
        ang += rng.normal(0.0, math.radians(4))
        qx, qy = px + seg * math.cos(ang), py + seg * math.sin(ang)
        cv2.line(canvas, (int(round(px)), int(round(py))), (int(round(qx)), int(round(qy))),
                 int(val), th, cv2.LINE_AA)
        px, py = qx, qy


def _canvas(layer_u8, sigma):
    return cv2.GaussianBlur(layer_u8, (0, 0), sigma).astype(np.float32) / 255.0


def _pick_free(rng, free, dist_mm, min_dist, tries=30):
    Hm, Wm = free.shape
    for _ in range(tries):
        x, y = int(rng.integers(0, Wm)), int(rng.integers(0, Hm))
        if free[y, x] > 0.9 and dist_mm[y, x] > min_dist:
            return x, y
    return None


# ---------------------------------------------------------------------
# Main entry: tex (H,W,3 linear float32), raised (H,W 0/1), item, rng -> (tex, info)
# ---------------------------------------------------------------------
def apply_wear(tex, raised, item, rng, wc=None):
    wc = wc or WEAR
    t0 = time.time()
    Hm, Wm = raised.shape
    if tex.shape[:2] != (Hm, Wm):                                       # works even if the ink block (which resizes) is off
        tex = cv2.resize(tex, (Wm, Hm), interpolation=cv2.INTER_LINEAR)
    tex = np.ascontiguousarray(tex, dtype=np.float32)
    bm = board_maps(item, (Hm, Wm), wc)
    texel, foot, dist_mm, cav = bm["texel_mm"], bm["foot"], bm["dist_mm"], bm["cav"]

    # guard: wear is allowed only where `free` > 0 (never on letters, never off the board)
    g_it = max(1, int(round(wc.guard_mm / texel)))
    guard = cv2.dilate(raised.astype(np.float32), np.ones((3, 3), np.uint8), iterations=g_it)
    free = np.clip(1.0 - cv2.GaussianBlur(guard, (0, 0), 1.0), 0.0, 1.0) * foot
    efade = _smooth(dist_mm, *wc.edge_fade_mm)                          # 0 at the outline, 1 from ~2.5 mm inward

    theta, coh = grain_angle(tex)
    if coh < wc.min_grain_coherence:
        theta = 0.0
    g = np.array([math.cos(theta), math.sin(theta)])
    area_cm2 = float(foot.sum()) * texel * texel / 100.0

    # ---------- 1. cracks ----------
    can_main = np.zeros((Hm, Wm), np.uint8)
    can_hair = np.zeros((Hm, Wm), np.uint8)
    n_edge = int(rng.integers(wc.n_edge_cracks[0], wc.n_edge_cracks[1] + 1))
    n_floor = int(rng.integers(wc.n_floor_cracks[0], wc.n_floor_cracks[1] + 1))
    cand, n_edge_done = bm["edge_cand"], 0
    if len(cand) and n_edge > 0:
        wgt = (cand[:, 2] * g[0] + cand[:, 3] * g[1]) ** 2 + 0.05       # favour end-grain edges (normal || grain)
        for j in rng.choice(len(cand), size=n_edge, p=wgt / wgt.sum()):
            cx, cy, nx_, ny_ = (float(v) for v in cand[j])
            inward = 0.0 if (nx_ * g[0] + ny_ * g[1]) >= 0 else math.pi  # run into the board, not out of it
            L = wc.crack_len_mm[0] + (wc.crack_len_mm[1] - wc.crack_len_mm[0]) * rng.random() ** 1.8
            _walk(can_main, cx, cy, theta + inward + rng.normal(0, math.radians(3)), L,
                  rng.uniform(*wc.crack_width_mm), 255 * rng.uniform(*wc.crack_dark),
                  rng, texel, wc.crack_wander_deg, wc.crack_branch_prob)
            n_edge_done += 1
    n_hair_done = 0
    for _ in range(n_floor):
        p = _pick_free(rng, free, dist_mm, 4.0)
        if p is None:
            continue
        L = wc.hairline_len_mm[0] + (wc.hairline_len_mm[1] - wc.hairline_len_mm[0]) * rng.random() ** 1.5
        flip = 0.0 if rng.random() < 0.5 else math.pi
        _walk(can_hair, p[0], p[1], theta + flip + rng.normal(0, math.radians(5)), L,
              rng.uniform(*wc.hairline_width_mm), 255 * rng.uniform(*wc.hairline_dark),
              rng, texel, wc.crack_wander_deg * 0.8, 0.0)
        n_hair_done += 1
    d_crack = _canvas(np.maximum(can_main, can_hair), 0.6) * efade * free
    del can_main, can_hair

    # ---------- 2. wear marks ----------
    n_scr = int(rng.uniform(*wc.scratch_per_cm2) * area_cm2)
    can_l = np.zeros((Hm, Wm), np.uint8)
    can_d = np.zeros((Hm, Wm), np.uint8)
    n_scr_done = 0
    for _ in range(n_scr):
        p = _pick_free(rng, free, dist_mm, 2.0, tries=10)
        if p is None:
            continue
        ang = theta + rng.normal(0, math.radians(wc.scratch_angle_jitter_deg)) if rng.random() < 0.8 \
            else rng.uniform(0, math.pi)
        _scratch(can_l if rng.random() < wc.scratch_light_frac else can_d, p[0], p[1], ang,
                 rng.uniform(*wc.scratch_len_mm), rng.uniform(*wc.scratch_width_mm),
                 rng.integers(110, 256), rng, texel)
        n_scr_done += 1
    s_amt = float(rng.uniform(*wc.scratch_strength))
    l_scr = _canvas(can_l, 0.6) * (s_amt * efade * free)
    d_scr = _canvas(can_d, 0.6) * (s_amt * efade * free)
    del can_l, can_d
    e_amt = float(rng.uniform(*wc.edge_wear_strength))                   # handled edges: patchy brownish darkening
    d_edge = e_amt * np.exp(-dist_mm / wc.edge_wear_band_mm) * (0.4 + 0.6 * _patchy(rng, (Hm, Wm), 5)) * free

    # ---------- 3. grime in recesses, dust film, specks ----------
    gr_amt = float(rng.uniform(*wc.grime_strength))
    w_grime = gr_amt * np.power(cav, wc.grime_gamma) * (0.5 + 0.5 * _patchy(rng, (Hm, Wm), 4)) * free
    du_amt = float(rng.uniform(*wc.dust_strength))
    w_dust = du_amt * _smooth(_patchy(rng, (Hm, Wm), 5), 0.45, 0.80) * free
    n_spk = int(rng.uniform(*wc.speck_per_cm2) * area_cm2)
    can_sd = np.zeros((Hm, Wm), np.uint8)
    can_sl = np.zeros((Hm, Wm), np.uint8)
    if n_spk > 0:
        M = n_spk * 4
        ys, xs = rng.integers(0, Hm, M), rng.integers(0, Wm, M)
        ok = (free[ys, xs] > 0.9) & (efade[ys, xs] > 0.5) & (rng.random(M) < 0.3 + 0.7 * cav[ys, xs])
        sel = np.flatnonzero(ok)[:n_spk]
        for j in sel:                                                    # specks are denser in recesses
            r = max(1, int(round(rng.uniform(*wc.speck_diam_mm) / texel / 2)))
            cv2.circle(can_sd if rng.random() < 0.6 else can_sl, (int(xs[j]), int(ys[j])), r,
                       int(rng.integers(120, 256)), -1, cv2.LINE_AA)
        n_spk = len(sel)
    sp_amt = float(rng.uniform(*wc.speck_strength))
    d_spk = _canvas(can_sd, 0.5) * (sp_amt * free)                       # specks are clipped by the guard too
    l_spk = _canvas(can_sl, 0.5) * (sp_amt * free)
    del can_sd, can_sl

    # ---------- 5. very light dust on letter tops ----------
    lt_amt = 0.0
    w_top = 0.0
    if wc.letter_dust:
        lt_amt = float(rng.uniform(*wc.letter_dust_strength))
        w_top = lt_amt * _smooth(_patchy(rng, (Hm, Wm), 5), 0.45, 0.80) * \
            cv2.GaussianBlur(raised.astype(np.float32), (0, 0), 0.8)

    # ---------- 4. age grade: draws come LAST so cracks/scratches/dust keep the exact same pattern per seed ----------
    ag = None
    if wc.age_grade:
        ag = (float(rng.uniform(*wc.age_desat)), float(rng.uniform(*wc.age_tint)),
              float(rng.uniform(*wc.age_mottle)), _patchy(rng, (Hm, Wm), 3))

    # ---------- compose (linear space) ----------
    sub = (slice(None, None, 5), slice(None, None, 5))                   # cheap subsample for the contrast bookkeeping
    lum0 = (tex[sub] @ _LUM); m_top, m_floor = raised[sub] > 0.5, free[sub] > 0.5
    ratio0 = float(lum0[m_top].mean() / max(lum0[m_floor].mean(), 1e-9)) if m_top.any() and m_floor.any() else float("nan")
    if ag is not None:                                                   # age grade goes under all the wear
        desat, tint_b, mot, nz = ag
        la = (tex @ _LUM)[..., None]
        graded = la + (tex - la) * (1.0 - desat)                         # remove colour, keep luminance
        graded = graded * (1.0 - tint_b) + la * _AGE_DIR * tint_b        # nudge toward grey-brown, keep luminance
        z = np.clip((nz - nz.mean()) / (nz.std() + 1e-6) / 2.5, -1.0, 1.0)   # soft stains: +/-1 at 2.5 sigma
        graded *= 1.0 + (mot * z)[..., None] * _TINT                     # darker patches go browner, lighter go yellower
        np.clip(graded, 0.0, 1.0, out=graded)
        if wc.age_on_letters:
            tex = graded
        else:
            tex += (graded - tex) * free[..., None]
        del graded, la, nz, z
    tex += (_GRIME - tex) * w_grime[..., None]                           # grime next
    lum = tex @ _LUM
    chalk = np.maximum(tex * 1.04, lum[..., None] * 1.12)                # only lightens / desaturates: dust never darkens
    tex += (chalk - tex) * np.clip(w_dust + l_spk + w_top, 0.0, 0.6)[..., None]
    d_tot = 1.0 - (1.0 - d_crack) * (1.0 - d_scr) * (1.0 - d_edge) * (1.0 - d_spk)
    tex *= np.clip(1.0 - d_tot[..., None] * _TINT, 0.03, 1.0)            # cracks, dark scratches, edge wear, dark specks
    tex *= (1.0 + l_scr)[..., None]                                      # light scratches (fresh wood)
    np.clip(tex, 0.0, 1.0, out=tex)
    lum1 = (tex[sub] @ _LUM)
    ratio1 = float(lum1[m_top].mean() / max(lum1[m_floor].mean(), 1e-9)) if m_top.any() and m_floor.any() else float("nan")

    info = {"grain_deg": round(math.degrees(theta) % 180.0, 1), "grain_coherence": round(float(coh), 3),
            "n_edge_cracks": n_edge_done, "n_hairlines": n_hair_done, "n_scratches": n_scr_done,
            "n_specks": int(n_spk), "scratch_strength": round(s_amt, 3), "edge_wear": round(e_amt, 3),
            "grime": round(gr_amt, 3), "dust": round(du_amt, 3), "letter_dust": round(lt_amt, 3),
            "age_desat": round(ag[0], 3) if ag else 0.0, "age_tint": round(ag[1], 3) if ag else 0.0,
            "age_mottle": round(ag[2], 3) if ag else 0.0,
            "letter_floor_lum_ratio_before": round(ratio0, 4), "letter_floor_lum_ratio_after": round(ratio1, 4),
            "time_s": round(time.time() - t0, 2)}
    return tex, info

def wear_preview(item, seed=0, crop_mm=(110, 70)):
    import matplotlib.pyplot as plt
    rng = np.random.default_rng(seed)
    tex, _ = load_ext_texture(rng)
    Hm, Wm = item["raised"].shape
    tex = cv2.resize(tex, (Wm, Hm), interpolation=cv2.INTER_LINEAR)
    after, info = apply_wear(tex.copy(), item["raised"], item, np.random.default_rng(seed + 2))
    texel = board_maps(item, (Hm, Wm), WEAR)["texel_mm"]
    cw, ch = int(crop_mm[0] / texel), int(crop_mm[1] / texel)
    views = {"whole board (1/8 scale)": (0, 0, Wm, Hm, 8),
             "centre": (Wm // 2 - cw // 2, Hm // 2 - ch // 2, cw, ch, 1),
             "left end": (int(8 / texel), Hm // 2 - ch // 2, cw, ch, 1)}
    fig, axs = plt.subplots(len(views), 2, figsize=(16, 5.2 * len(views)))
    for r, (name, (x0, y0, w, h, ds)) in enumerate(views.items()):
        for c, (lab, im) in enumerate((("before", tex), ("after", after))):
            v = im[y0:y0 + h, x0:x0 + w]
            if ds > 1:
                v = cv2.resize(v, (w // ds, h // ds), interpolation=cv2.INTER_AREA)
            axs[r, c].imshow(np.clip(srgb(v), 0, 1)); axs[r, c].set_title(f"{name} - {lab}"); axs[r, c].axis("off")
    # MỚI
    plt.tight_layout(); plt.savefig(ROOT / "selftest" / "wear_preview.png", dpi=90); plt.show()
    print(info)

## 12.4. Ba profile quality × hai level độ nét
Mỗi profile CLEAN / NORMAL / HARD có hai level `sharp` và `soft`. Level được gán ổn định, cân bằng trong từng profile theo số ảnh có thể chia (ví dụ 20 ảnh: CLEAN 5+5, NORMAL 3+2, HARD 3+2). `data_profile` vẫn là nhãn chất lượng chính; `detail_level` là nhãn độ nét bên trong profile. CLEAN luôn giữ noise bằng 0, camera gần trực diện, fill sáng và QA nghiêm ngặt; HARD giữ nguyên dải góc/ánh sáng/nhiễu cơ bản hiện tại.

In [ ]:
# CELL A — CLEAN: camera chuẩn trực diện + đèn rọi xiên thấp để nổi relief chữ
# Cả hai level CLEAN đều giữ: không noise, môi trường sáng, chữ phải vượt QA độ nét + tương phản.
CLEAN_CONFIG = {
    "camera_geometry": {
        "tilt_deg": (0.0, 2.0),
        "roll_deg": (-0.5, 0.5),
        "azim_mode": "configured",
    },
    "lighting": {
        # Không dùng camera_ring cũ: nó có thể đặt đèn quá cao, làm phẳng bóng relief.
        # Đèn raking ở 12–22° so với mặt ván giúp tạo bóng ngắn, rõ cạnh nét khắc.
        "mode": "raking",
        "elev_deg": (12.0, 22.0),
        "fill_ratio": (0.12, 0.18),
        "size_ratio": (0.025, 0.045),
        "key_power_multiplier": (0.95, 1.05),
        "second_light_prob": 0.0,
    },
    "exposure_target": (0.47, 0.58),
    "qa_min_text_sharpness": 4.0,
    "qa_min_shading_contrast": 0.22,
    "camera_effects": {
        "wb_gain": (0.99, 1.01),
        "vignette": (0.0, 0.0),
        "defocus_sigma_px": (0.0, 0.0),
        "motion_blur_prob": 0.0,
        "noise_a": (0.0, 0.0),
        "noise_b": (0.0, 0.0),
        "jpeg_quality": (98, 100),
        "sharpen_skip_prob": 0.0,
        "sharpen_amount": (0.25, 0.45),
        "sharpen_sigma_px": (0.70, 1.05),
    },
    "detail_levels": {
        "sharp": {
            "camera_effects": {
                "defocus_sigma_px": (0.0, 0.0),
                "sharpen_amount": (0.45, 0.70),
                "sharpen_sigma_px": (0.55, 0.95),
                "jpeg_quality": (99, 100),
                "noise_a": (0.0, 0.0),
                "noise_b": (0.0, 0.0),
                "motion_blur_prob": 0.0,
                "vignette": (0.0, 0.0),
            },
            "qa_min_text_sharpness": 5.0,
            "qa_min_shading_contrast": 0.24,
        },
        "soft": {
            # Mờ rất nhẹ để có level thứ hai, nhưng vẫn phải qua cổng QA CLEAN.
            "camera_effects": {
                "defocus_sigma_px": (0.21, 0.28),
                "sharpen_amount": (0.18, 0.34),
                "sharpen_sigma_px": (0.70, 1.00),
                "jpeg_quality": (97, 99),
                "noise_a": (0.0, 0.0),
                "noise_b": (0.0, 0.0),
                "motion_blur_prob": 0.0,
                "vignette": (0.0, 0.0),
            },
            "qa_min_text_sharpness": 4.0,
            "qa_min_shading_contrast": 0.22,
        },
    },
}
print("CLEAN_CONFIG ready: tilt <= 2°, low-angle raking light, bright fill, zero noise; sharp/soft both QA-gated")


In [ ]:
# CELL B — NORMAL: mix giữa CLEAN và HARD, có 2 level nét và suy giảm nhẹ có kiểm soát
NORMAL_CONFIG = {
    "camera_geometry": {
        "tilt_deg": (2.1, 20.0),
        "roll_deg": (-4.0, 4.0),
        "azim_mode": "configured",
    },
    "lighting": {
        "mode": "random",
        "elev_deg": (14.0, 34.0),
        "fill_ratio": (0.035, 0.075),
        "size_ratio": (0.07, 0.13),
        "key_power_multiplier": (0.55, 0.75),
        "second_light_prob": 0.12,
    },
    "exposure_target": (0.31, 0.44),
    "qa_min_text_sharpness": 2.0,
    "qa_min_shading_contrast": 0.18,
    "camera_effects": {
        "wb_gain": (0.97, 1.03),
        "vignette": (0.0, 0.06),
        "defocus_sigma_px": (0.0, 0.20),
        "motion_blur_prob": 0.02,
        "noise_a": (0.0, 0.00055),
        "noise_b": (0.0, 5e-6),
        "jpeg_quality": (93, 99),
        "sharpen_skip_prob": 0.28,
        "sharpen_amount": (0.25, 0.48),
        "sharpen_sigma_px": (0.60, 1.10),
    },
    "detail_levels": {
        "sharp": {
            "camera_effects": {
                "defocus_sigma_px": (0.0, 0.12),
                "motion_blur_prob": 0.01,
                "noise_a": (0.00005, 0.00035),
                "noise_b": (0.0, 3e-6),
                "vignette": (0.0, 0.04),
                "sharpen_amount": (0.32, 0.52),
                "jpeg_quality": (95, 99),
            },
            "qa_min_text_sharpness": 2.8,
            "qa_min_shading_contrast": 0.18,
        },
        "soft": {
            "camera_effects": {
                "defocus_sigma_px": (0.23, 0.42),
                "motion_blur_prob": 0.04,
                "noise_a": (0.00045, 0.0010),
                "noise_b": (3e-6, 1e-5),
                "vignette": (0.025, 0.07),
                "sharpen_amount": (0.15, 0.34),
                "jpeg_quality": (90, 96),
            },
            "qa_min_text_sharpness": 1.8,
            "qa_min_shading_contrast": 0.16,
        },
    },
}
print("NORMAL_CONFIG ready: slightly darker than CLEAN, light/noise/blur vary by sharp/soft level")


In [ ]:
# CELL C — HARD: giữ nguyên hình học, ánh sáng, noise và hiệu ứng nền HARD hiện tại.
# Hai level chỉ phân tầng độ sắc/mờ hậu kỳ; camera + đèn của HARD không bị đổi.
HARD_CONFIG = {
    "camera_geometry": {
        "tilt_deg": (20.0, 60.0),
        "roll_deg": (-6.0, 6.0),
        "azim_mode": "full",
    },
    "lighting": {
        "mode": "random",
        "elev_deg": (5.0, 32.0),
        "fill_ratio": (0.006, 0.025),
        "size_ratio": (0.04, 0.09),
        "key_power_multiplier": (0.25, 0.50),
        "second_light_prob": 0.05,
    },
    "exposure_target": (0.17, 0.30),
    "qa_min_text_sharpness": 0.75,
    "qa_min_shading_contrast": 0.18,
    "camera_effects": {
        "wb_gain": (0.88, 1.12),
        "vignette": (0.06, 0.25),
        "defocus_sigma_px": (0.0, 0.65),
        "motion_blur_prob": 0.14,
        "noise_a": (0.0010, 0.0040),
        "noise_b": (1.2e-5, 5e-5),
        "jpeg_quality": (72, 88),
        "sharpen_skip_prob": 0.50,
        "sharpen_amount": (0.20, 0.45),
        "sharpen_sigma_px": (0.70, 1.20),
    },
    "detail_levels": {
        "sharp": {
            # Không override camera, lighting, blur, noise, JPEG hay sharpen base của HARD.
            "qa_min_text_sharpness": 0.75,
            "qa_min_shading_contrast": 0.18,
        },
        "soft": {
            # Chỉ hạ mức sharpen để tạo level mềm hơn; toàn bộ điều kiện HARD còn lại giữ nguyên.
            "camera_effects": {
                "sharpen_amount": (0.10, 0.25),
                "sharpen_skip_prob": 0.70,
            },
            "qa_min_text_sharpness": 0.55,
            "qa_min_shading_contrast": 0.18,
        },
    },
}
print("HARD_CONFIG ready: baseline camera/light/noise preserved; split only by relative sharpness level")


In [ ]:
# Shared registry: các thông số được khai báo độc lập ở 3 cell phía trên.
import zlib
from dataclasses import replace as dataclass_replace

PROFILE_WEIGHTS = {"clean": 0.50, "normal": 0.25, "hard": 0.25}
PROFILE_LEVELS = ("sharp", "soft")
PROFILE_CONFIGS = {"clean": CLEAN_CONFIG, "normal": NORMAL_CONFIG, "hard": HARD_CONFIG}
# Base maps retained for diagnostics/compatibility; effective values come from get_profile_config().
CAMERA_GEOMETRY = {name: cfg_item["camera_geometry"] for name, cfg_item in PROFILE_CONFIGS.items()}
LIGHT_PROFILE = {name: cfg_item["lighting"] for name, cfg_item in PROFILE_CONFIGS.items()}
CAMERA_PROFILE = {name: cfg_item["camera_effects"] for name, cfg_item in PROFILE_CONFIGS.items()}
EXPOSURE_PROFILE = {name: cfg_item["exposure_target"] for name, cfg_item in PROFILE_CONFIGS.items()}
QA_PROFILE = {name: cfg_item["qa_min_text_sharpness"] for name, cfg_item in PROFILE_CONFIGS.items()}


def get_profile_config(profile, detail_level="sharp"):
    """Hợp nhất cấu hình nền của profile với override theo level sharp/soft."""
    if profile not in PROFILE_CONFIGS:
        raise ValueError(f"Unknown data profile: {profile}")
    if detail_level not in PROFILE_LEVELS:
        raise ValueError(f"Unknown detail level: {detail_level}")
    base = PROFILE_CONFIGS[profile]
    result = {
        key: (value.copy() if isinstance(value, dict) else value)
        for key, value in base.items()
        if key != "detail_levels"
    }
    overrides = base.get("detail_levels", {}).get(detail_level, {})
    for key, value in overrides.items():
        if isinstance(value, dict) and isinstance(result.get(key), dict):
            merged = result[key].copy()
            merged.update(value)
            result[key] = merged
        else:
            result[key] = value
    return result


def measure_text_sharpness_rgb(image_rgb, chars):
    """Laplacian variance trong polygon ký tự thấy được trên ảnh cuối đã JPEG-encode/decode."""
    if image_rgb is None:
        return None, "unavailable"
    gray = cv2.cvtColor(np.asarray(image_rgb, dtype=np.uint8), cv2.COLOR_RGB2GRAY)
    H, W = gray.shape[:2]
    lap = cv2.Laplacian(gray, cv2.CV_32F, ksize=3)
    usable = [c for c in (chars or []) if c.get("visible") and not c.get("pseudo") and c.get("poly_px")]
    if len(usable) > 256:
        ids = np.linspace(0, len(usable) - 1, 256).astype(int)
        usable = [usable[k] for k in ids]
    vals = []
    for ch in usable:
        try:
            poly = np.asarray(ch["poly_px"], dtype=np.float32).reshape(-1, 2)
            if len(poly) < 3 or not np.isfinite(poly).all():
                continue
            x0 = max(0, int(math.floor(float(poly[:, 0].min()))))
            y0 = max(0, int(math.floor(float(poly[:, 1].min()))))
            x1 = min(W, int(math.ceil(float(poly[:, 0].max()))) + 1)
            y1 = min(H, int(math.ceil(float(poly[:, 1].max()))) + 1)
            if x1 - x0 < 4 or y1 - y0 < 4:
                continue
            local_poly = np.round(poly - np.array([x0, y0], dtype=np.float32)).astype(np.int32)
            region_mask = np.zeros((y1-y0, x1-x0), dtype=np.uint8)
            cv2.fillPoly(region_mask, [local_poly], 255)
            pixels = lap[y0:y1, x0:x1][region_mask > 0]
            if pixels.size >= 12:
                val = float(np.var(pixels))
                if math.isfinite(val):
                    vals.append(val)
        except (ValueError, TypeError, IndexError):
            continue
    if vals:
        return float(np.median(vals)), "visible_character_regions"
    # Fallback chỉ cho mesh không có polygon ký tự thực; không dùng làm ngưỡng QA OCR.
    small = cv2.resize(gray, (max(8, W // 2), max(8, H // 2)), interpolation=cv2.INTER_AREA)
    return float(np.var(cv2.Laplacian(small, cv2.CV_32F, ksize=3))), "whole_image_fallback"


def get_profile_value(profile, key, detail_level="sharp"):
    """Trả về thông số theo profile và level độ nét."""
    effective = get_profile_config(profile, detail_level)
    if key == "exposure_target":
        return effective["exposure_target"]
    value = effective["camera_effects"].get(key)
    return value if value is not None else getattr(cfg, key)


# ------------------------------------------------------------
# Gán profile xác định, có thể tái lập; n=20 -> 10/5/5.
# ------------------------------------------------------------
_PROFILE_SCHEDULE_CACHE = {}

def make_profile_schedule(item_tag, n):
    if n < 1:
        raise ValueError("Số ảnh phải lớn hơn 0")
    key = (item_tag, n)
    if key in _PROFILE_SCHEDULE_CACHE:
        return _PROFILE_SCHEDULE_CACHE[key]
    raw_counts = {name: n * weight for name, weight in PROFILE_WEIGHTS.items()}
    counts = {name: int(raw_counts[name]) for name in PROFILE_WEIGHTS}
    remaining = n - sum(counts.values())
    ranked = sorted(PROFILE_WEIGHTS, key=lambda name: raw_counts[name] - counts[name], reverse=True)
    for name in ranked[:remaining]:
        counts[name] += 1
    schedule = [name for name in PROFILE_WEIGHTS for _ in range(counts[name])]
    seed = cfg.seed + zlib.crc32(item_tag.encode("utf-8")) + n * 97
    rng = np.random.default_rng(seed)
    rng.shuffle(schedule)
    _PROFILE_SCHEDULE_CACHE[key] = schedule
    return schedule


def get_data_profile(item_tag, i, n):
    """Profile lúc render, ổn định khi chạy lại cùng cấu hình."""
    return make_profile_schedule(item_tag, n)[i % n]


_DETAIL_SCHEDULE_CACHE = {}

def make_profile_detail_schedule(item_tag, n):
    """Chia sharp/soft gần 50/50 BÊN TRONG từng profile, có seed ổn định."""
    key = (item_tag, n)
    if key in _DETAIL_SCHEDULE_CACHE:
        return _DETAIL_SCHEDULE_CACHE[key]
    profiles = make_profile_schedule(item_tag, n)
    levels = [None] * n
    for profile in PROFILE_WEIGHTS:
        indices = [i for i, assigned in enumerate(profiles) if assigned == profile]
        seed = cfg.seed + zlib.crc32(f"{item_tag}|{profile}|detail".encode("utf-8")) + n * 139
        rng = np.random.default_rng(seed)
        rng.shuffle(indices)
        first = int(rng.integers(0, 2))
        for rank, idx in enumerate(indices):
            levels[idx] = PROFILE_LEVELS[(rank + first) % len(PROFILE_LEVELS)]
    _DETAIL_SCHEDULE_CACHE[key] = levels
    return levels


def get_detail_level(item_tag, i, n):
    """Level độ nét của ảnh; cân bằng riêng bên trong từng profile."""
    return make_profile_detail_schedule(item_tag, n)[i % n]


def wear_config_for_profile(profile):
    if profile == "clean":
        return WEAR

    if profile == "normal":
        # Mức hao mòn nhẹ: kế thừa cấu hình clean cũ.
        return dataclass_replace(
            WEAR,

            n_edge_cracks=(0, 1),
            n_floor_cracks=(0, 2),

            scratch_per_cm2=(0.01, 0.04),
            scratch_strength=(0.03, 0.08),
            edge_wear_strength=(0.02, 0.05),

            grime_strength=(0.02, 0.08),
            dust_strength=(0.02, 0.08),
            speck_per_cm2=(0.05, 0.20),

            letter_dust_strength=(0.01, 0.02),

            age_desat=(0.02, 0.08),
            age_tint=(0.01, 0.05),
            age_mottle=(0.01, 0.04),
        )

    if profile == "hard":
        return dataclass_replace(
            WEAR,

            n_edge_cracks=(4, 9),
            n_floor_cracks=(6, 14),

            scratch_per_cm2=(0.10, 0.20),
            scratch_strength=(0.14, 0.30),
            edge_wear_strength=(0.12, 0.24),

            grime_strength=(0.14, 0.30),
            dust_strength=(0.17, 0.38),
            speck_per_cm2=(0.60, 1.70),

            letter_dust_strength=(0.04, 0.08),

            age_desat=(0.22, 0.42),
            age_tint=(0.16, 0.27),
            age_mottle=(0.09, 0.15),
        )

    raise ValueError(f"Unknown data profile: {profile}")

print("Profiles:", PROFILE_WEIGHTS, "| levels:", PROFILE_LEVELS)
_demo_profiles = make_profile_schedule("test", 20)
_demo_levels = make_profile_detail_schedule("test", 20)
print("Số ảnh mỗi profile nếu n=20:", {k: _demo_profiles.count(k) for k in PROFILE_WEIGHTS})
print("Số level mỗi profile nếu n=20:", {
    p: {level: sum(1 for i, assigned in enumerate(_demo_profiles)
                   if assigned == p and _demo_levels[i] == level)
        for level in PROFILE_LEVELS}
    for p in PROFILE_WEIGHTS
})

# Guardrails: profile boundaries and the user's core CLEAN/HARD constraints.
assert CLEAN_CONFIG["camera_geometry"]["tilt_deg"][1] <= 2.0
assert CLEAN_CONFIG["lighting"]["mode"] == "raking"
assert max(CLEAN_CONFIG["lighting"]["elev_deg"]) <= 22.0
assert CLEAN_CONFIG["camera_effects"]["noise_a"] == (0.0, 0.0)
assert CLEAN_CONFIG["camera_effects"]["noise_b"] == (0.0, 0.0)
assert all(CLEAN_CONFIG["detail_levels"][level]["camera_effects"]["noise_a"] == (0.0, 0.0)
           and CLEAN_CONFIG["detail_levels"][level]["camera_effects"]["noise_b"] == (0.0, 0.0)
           for level in PROFILE_LEVELS)
assert HARD_CONFIG["camera_geometry"]["tilt_deg"][0] >= 20.0
assert max(HARD_CONFIG["lighting"]["key_power_multiplier"]) <= 0.50
assert max(HARD_CONFIG["lighting"]["fill_ratio"]) < min(NORMAL_CONFIG["lighting"]["fill_ratio"])
assert max(NORMAL_CONFIG["lighting"]["fill_ratio"]) < min(CLEAN_CONFIG["lighting"]["fill_ratio"])
assert max(HARD_CONFIG["exposure_target"]) < min(NORMAL_CONFIG["exposure_target"])
assert max(NORMAL_CONFIG["exposure_target"]) < min(CLEAN_CONFIG["exposure_target"])
assert max(NORMAL_CONFIG["lighting"]["key_power_multiplier"]) < min(CLEAN_CONFIG["lighting"]["key_power_multiplier"])
print("Profile guardrails: PASS (CLEAN bright/raking/zero noise; NORMAL intermediate; HARD base camera/light retained)")


## 12.5 Research output contract và metric engine

Mã tham chiếu tách khỏi renderer. GT và proxy có provenance riêng; không fallback toàn vật thể cho E nét.


In [ ]:
# Single metric implementation, loaded after section 12.5 and before section 13.
PATCH_PATH = PROJECT_ROOT / "v5_metrics_update.py"
exec(compile(PATCH_PATH.read_text(encoding="utf-8"), str(PATCH_PATH), "exec"), globals())
print("Metrics backend:", cfg.metric_backend_path)


## 13. Render một ảnh
Bốc tham số → render → kiểm tra độ phủ, clipping, tương phản chữ và QA trên JPEG cuối (sharpness vùng ký tự). Nếu ảnh cuối lỗi rõ rệt, bốc lại tối đa `max_tries` lần; lý do được ghi vào JSON. Mỗi ảnh có seed riêng suy ra từ `cfg.seed` nên chạy lại cho kết quả y hệt; ảnh đã có sẽ được bỏ qua (chạy tiếp được sau khi ngắt).


**bổ sung phần WEAR từ 12.3 :texture gỗ (đã nâng lên lưới 0.1 mm) + mặt nạ chữ nổi -->  Kiểm tra điều kiện -->  Chuẩn bị --> Tạo lớp --> Ghép vào texture --> Ghi log + render**



In [ ]:
def render_one(
    i, item, W=None, H=None, spp=None,
    data_profile=None, detail_level=None,
):
    W, H, spp = W or cfg.width, H or cfg.height, spp or SPP
    name = f"{item['tag']}_{i:05d}"
    
    if data_profile is None:
        data_profile = get_data_profile(item["tag"], i, cfg.n_images)
    if detail_level is None:
        detail_level = get_detail_level(item["tag"], i, cfg.n_images)
    global _ACTIVE_CONDITION
    assigned_profile, assigned_level, assigned_condition, condition = experiment_assignment(item['tag'],i,cfg.n_images)
    if cfg.experiment_mode=='controlled':
        data_profile,detail_level=assigned_profile,assigned_level
    _ACTIVE_CONDITION=condition
    profile_settings = get_profile_config(data_profile, detail_level)
    generation_profile = f"{data_profile}_{detail_level}"
    pipeline_version = PIPELINE_VERSION
    fingerprint = sample_signature(item,cfg,profile_settings)

    wear_cfg = wear_config_for_profile(data_profile)
    jpath = ROOT / "labels" / f"{name}.json"
    if jpath.exists():
        # Bỏ qua chỉ khi JSON accepted VÀ file ảnh còn tồn tại.
        # JSON bị reject/stale được render lại ở lần chạy sau, thay vì bị skip vĩnh viễn.
        try:
            previous = json.loads(jpath.read_text(encoding="utf-8"))
            image_rel = previous.get("image")
            image_exists = bool(image_rel and (ROOT / image_rel).exists())
            log_path=ROOT/"logs"/f"{name}_metric.json"
            if log_path.exists():
                logged=json.loads(log_path.read_text(encoding="utf-8"))
                image_exists=image_exists and all((ROOT/p).exists() for p in logged.get('files',{}).values())
            else:image_exists=False
            if (previous.get("accepted") and image_exists
                    and previous.get("pipeline_version") == pipeline_version
                    and previous.get("experiment_fingerprint") == fingerprint
                    and (ROOT / "logs" / f"{name}_metric.json").exists()
                    and previous.get("generation_profile") == generation_profile
                    and previous.get("detail_level") == detail_level):
                return None
            if log_path.exists():
                for rel in logged.get('files',{}).values():
                    stale=(ROOT/rel).resolve()
                    if ROOT.resolve() in stale.parents:stale.unlink(missing_ok=True)
            # Nếu profile/level hoặc phiên bản pipeline đã đổi, dọn artefact cũ để tránh output rác.
            old_image = previous.get("image")
            if old_image:
                (ROOT / old_image).unlink(missing_ok=True)
                old_stem = Path(old_image).stem
                (ROOT / "overlays" / f"{old_stem}.jpg").unlink(missing_ok=True)
            for old_patch in previous.get("patches", []):
                old_patch_path = old_patch.get("file")
                if old_patch_path:
                    (ROOT / old_patch_path).unlink(missing_ok=True)
            jpath.unlink(missing_ok=True)
        except Exception:
            jpath.unlink(missing_ok=True)
    # Controlled conditions in the same repetition use the SAME RNG seed for pairing.
    seed_index = i // len(cfg.controlled_conditions) if cfg.experiment_mode=='controlled' else i
    seed = int(cfg.seed + 7919 * seed_index + zlib.crc32(item["tag"].encode()) % 100003)   # hash() của Python không cố định giữa các lần chạy
    rng = np.random.default_rng(seed)
    (ROOT / "rejected" / f"{name}.json").unlink(missing_ok=True)
    (ROOT / "logs" / f"{name}_metric.json").unlink(missing_ok=True)
    reasons, t0 = [], time.time()
    
    cover = contrast = enh_info = einfo = None
    
    for attempt in range(cfg.max_tries):
        cam, cparams = sample_camera(item["mesh"], rng, W, H, item["plate"], data_profile=data_profile, detail_level=detail_level)
        tex, tinfo = load_ext_texture(rng) if getattr(cfg, "texture_paths", []) else make_wood_texture(rng)
        
        if item["raised"] is not None and cfg.ink_enabled:
            Hm, Wm = item["raised"].shape
            tex = cv2.resize(tex, (Wm, Hm), interpolation=cv2.INTER_LINEAR)       # nâng texture lên cùng độ phân giải
            ink = rng.uniform(*cfg.ink_strength)
            noise = cv2.resize(fractal_noise(rng, 1024, octaves=4), (Wm, Hm), interpolation=cv2.INTER_CUBIC)
            patchy = (noise > rng.uniform(*cfg.ink_patch_thresh)).astype(np.float32)
            layer = cv2.GaussianBlur(item["raised"] * patchy, (0, 0), cfg.ink_edge_sigma_px)
            tex = tex * (1 - ink * layer)[..., None]
            tinfo["ink_strength"] = ink

        
        # MODIFIED - KẾT HỢP RENDER_ONE CỦA GỖ BAN ĐẦU VỚI WEAR
        # Thêm vết cũ kỹ (nứt, xước, bụi, bẩn, màu già) vào texture gỗ trước khi render.
        # Nó chỉ đổi màu bề mặt, không đụng vào ánh sáng hay độ tương phản.
        # không đặt gì lên chữ ngoài lớp bụi rất nhẹ.
        
        
        if wear_cfg.enabled and item["raised"] is not None:
            tex, winfo = apply_wear(
                tex,
                item["raised"],
                item,
                np.random.default_rng(seed + 2),
                wear_cfg,
            )
            tinfo["wear"] = winfo    

    
        tpath = ROOT / "assets" / "_tex.png"
        cv2.imwrite(str(tpath), cv2.cvtColor((srgb(tex) * 255).astype(np.uint8), cv2.COLOR_RGB2BGR),
            [cv2.IMWRITE_PNG_COMPRESSION, 1])
        rough = rng.uniform(0.4, 0.8)
        bg = "white" if rng.random() < 0.4 else rng.choice(["light_gray", "dark_cloth", "wood_table"])
        ls = light_scale(item["mesh"])
        lights, linfo = sample_lights_outside(rng, cam, ls, data_profile=data_profile, detail_level=detail_level)
        if lights is None:
            reasons.append("light always in frame"); continue
        scene = build_scene(cam, item["ply"], tpath, rough, bg, lights, spp, item["zmin"], gs=4.0 * ls)
        hdr = np.array(mi.render(scene, spp=spp, seed=seed + attempt))[..., :3]
        depth, mask = render_depth(cam, item["ply"])
        cover = float(mask.mean())
        min_cov = cfg.min_coverage_plate if item["plate"] else cfg.min_coverage_other
        if cover < min_cov:
            reasons.append(f"coverage {cover:.2f} < {min_cov}"); continue
        # --- tự phơi sáng: trung vị độ sáng vật thể về mục tiêu, chặn trên để không cháy ---
        Ylin = luminance(hdr)
        target = rng.uniform(*get_profile_value(data_profile, "exposure_target", detail_level=detail_level))
        med = float(np.median(Ylin[mask]))
        k = float(srgb_inv(target) / max(med, 1e-8))
        p995 = float(np.quantile(hdr[mask].max(1), 0.995))
        k = min(k, 0.97 / max(p995, 1e-8))
        lin = hdr * k
        sat = float((lin[mask].max(1) >= 1.0).mean())
        if sat > cfg.max_saturated_frac:
            reasons.append(f"saturated {sat:.3f}"); continue
            
        ldr = srgb(lin)      

        occ = occlusion_scene(item["ply"])
            
        chars_px = project_labels(item["chars"], cam, occ)
        cols_px = project_labels(item["cols"], cam, occ)

        contrast = None
        if item["has_text"]:                      # CŨ: if item["chars"]:  (gồm cả nhãn giả)
            alb = render_albedo(cam, item["ply"], tpath, item["zmin"], BACKGROUNDS[bg])
            contrast = shading_contrast(hdr, alb, mask, chars_px)
            min_contrast = float(profile_settings.get("qa_min_shading_contrast", cfg.min_shading_contrast))
            if not BENCHMARK_MODE and contrast is not None and contrast < min_contrast:
                reasons.append(f"low shading contrast {contrast:.3f} < {min_contrast:.3f}"); continue
        elif item["plate"]:                       # ván thật chưa có nhãn: chỉ GHI LẠI, chưa loại
            alb = render_albedo(cam, item["ply"], tpath, item["zmin"], BACKGROUNDS[bg])
            Ls = luminance(hdr) / np.maximum(luminance(alb), 1e-4)
            v = Ls[mask]; lo, hi = np.percentile(v, [2, 98]); v = v[(v >= lo) & (v <= hi)]
            contrast = float(v.std() / max(v.mean(), 1e-6))

        # Phải QA ảnh CUỐI (sau hiệu ứng camera + JPEG), không chỉ HDR trước hậu kỳ.
        buf, einfo = camera_effects(
            ldr, rng, np.random.default_rng(seed + 1 + attempt),
            data_profile=data_profile, detail_level=detail_level
        )
        decoded = cv2.imdecode(buf, cv2.IMREAD_COLOR)
        if decoded is None:
            reasons.append("final JPEG decode failed")
            continue
        final = cv2.cvtColor(decoded, cv2.COLOR_BGR2RGB)
        final_hi_clip = float(((final.max(axis=2) >= 254) & mask).sum() / max(int(mask.sum()), 1))
        if final_hi_clip > cfg.max_saturated_frac:
            reasons.append(f"final JPEG clipped {final_hi_clip:.3f}")
            continue

        final_sharpness, sharpness_source = measure_text_sharpness_rgb(final, chars_px)
        min_sharpness = float(profile_settings["qa_min_text_sharpness"])
        if not BENCHMARK_MODE and item["has_text"] and final_sharpness is not None and final_sharpness < min_sharpness:
            reasons.append(f"final text sharpness {final_sharpness:.3f} < {min_sharpness:.3f}")
            continue
        final_qa = {
            "text_sharpness_laplacian_var": final_sharpness,
            "sharpness_source": sharpness_source,
            "min_text_sharpness": min_sharpness,
            "min_shading_contrast": float(profile_settings.get("qa_min_shading_contrast", cfg.min_shading_contrast)),
            "shading_contrast": contrast,
            "detail_level": detail_level,
            "high_clip_frac_on_object": final_hi_clip,
            "passed": True,  # Đạt các gate đang bật; xem benchmark_mode/gates.
            "benchmark_mode": BENCHMARK_MODE,
            "contrast_sharpness_gates_enabled": not BENCHMARK_MODE,
        }
        break
    
    else:
        rec = {
            "image": None,
            "object": item["tag"],
            "seed": seed,
            "data_profile": data_profile,
            "render_profile": data_profile,
            "detail_level": detail_level,
            "generation_profile": generation_profile,
            "pipeline_version": pipeline_version,
            "experiment_fingerprint": fingerprint,"assigned_condition":assigned_condition,
            "run_id":RUN_ID,"sample_id":name,"board_id":item['board_id'],
            "accepted": False,
            "rejections": reasons,
            "coverage": cover,
            "shading_contrast": contrast,
            "effects": einfo,
            "enhance": enh_info,
        }
        jpath.write_text(json.dumps(rec, ensure_ascii=False, indent=2), encoding="utf-8")
        (ROOT / "rejected" / f"{name}.json").write_text(
            json.dumps(rec, ensure_ascii=False, indent=2), encoding="utf-8"
        )
        export_rejection(ROOT,rec)
        return rec

    sample_metadata=dict(seed=seed,assigned_condition=assigned_condition,observed_quality=None,
                         pair_id=(RUN_ID+'_'+item['board_id']+'_'+str(seed_index)) if cfg.experiment_mode=='controlled' else None,
                         experiment_mode=cfg.experiment_mode,experiment_fingerprint=fingerprint,
                         camera={**cparams,"K":cam['K'],"R":cam['R'],"t":cam['t']},
                         lights=linfo,effects=einfo,material={**tinfo,"roughness":rough},
                         exposure={"target_srgb":target,"scale":k},spp=spp,resolution=[W,H],
                         retry_count=attempt,rejection_history=reasons,
                         render_and_export_time_s=time.time()-t0)
    render_rgb=(np.clip(ldr,0,1)*255+.5).astype(np.uint8)
    image_metrics = benchmark_render_metrics(final,item,cam,chars_px,ROOT,name,render_rgb,sample_metadata)
    (ROOT / "images" / f"{name}.jpg").write_bytes(buf.tobytes())

    #Lấy ảnh ldr hiện tại và chuyển nó thành ảnh RGB 8-bit để dùng làm nguồn crop/warp cho patch OCR.
    patches = []
    # OCR patches require verified transcription as well as geometric labels.
    if item["has_text"] and item.get('text_validated'):
        native = cam["K"][0, 0] / (cparams["distance_m"] * 1000.0)   # px/mm thật của ảnh render
        ppm = min(cfg.patch_px_per_mm, native)                        # không phóng đại quá độ phân giải thật
        for c, c3 in zip(cols_px, item["cols"]):
            if not c["visible"]:
                continue
            q3 = np.asarray(c3["quad"])
            wmm = np.linalg.norm(q3[1] - q3[0]) * 1000; hmm = np.linalg.norm(q3[3] - q3[0]) * 1000
            pw, ph = max(8, int(wmm * ppm)), max(8, int(hmm * ppm))
            src = np.asarray(c["poly_px"], np.float32) - 0.5
            dst = np.float32([[0, 0], [pw - 1, 0], [pw - 1, ph - 1], [0, ph - 1]])
            patch = cv2.warpPerspective(final, cv2.getPerspectiveTransform(src, dst), (pw, ph),
                                        flags=cv2.INTER_CUBIC)   # final = ảnh sau camera_effects + JPEG
            pn = f"{name}_col{c['col']}.jpg"
            cv2.imwrite(str(ROOT / "patches" / pn), cv2.cvtColor(patch, cv2.COLOR_RGB2BGR))
            patches.append({"file": f"patches/{pn}", "text": c["text"]})

    # overlay kiểm tra: xanh = nhìn thấy, đỏ = bị che/ra ngoài khung
    ov = cv2.cvtColor(final, cv2.COLOR_RGB2BGR)
    for c in chars_px:
        col = (0, 200, 0) if c["visible"] else (0, 0, 255)
        cv2.polylines(ov, [np.round(np.asarray(c["poly_px"]) - 0.5).astype(np.int32)], True, col, 1)
    for c in cols_px:
        col = (0, 220, 255) if c["visible"] else (0, 0, 255)
        p = np.round(np.asarray(c["poly_px"]) - 0.5).astype(np.int32)
        cv2.polylines(ov, [p], True, col, 2)
        cv2.putText(ov, str(c["col"]), tuple(p[0]), cv2.FONT_HERSHEY_SIMPLEX, 0.6, col, 2)
    if not chars_px:
        cnts, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(ov, cnts, -1, (255, 255, 0), 2)
    cv2.imwrite(str(ROOT / "overlays" / f"{name}.jpg"), ov)

    #====================================================================
    #BỔ SUNG DATA PROFILE VÀO JSON ĐỂ LƯU LẠI SỐ LƯỢNG CÁC FILE THEO LOẠI
    #====================================================================
    rec = {"image": f"images/{name}.jpg", "object": item["tag"], "seed": seed,
           "data_profile": data_profile, "render_profile": data_profile,
           "detail_level": detail_level, "generation_profile": generation_profile, "pipeline_version": pipeline_version,
           "accepted": True,
           "sample_id":name,"board_id":item['board_id'],"mesh_id":item['mesh_id'],
           "assigned_condition":assigned_condition,"observed_quality":None,
           "experiment_fingerprint":fingerprint,"annotation_version":item.get('annotation_version'),
           "attempts": attempt + 1, "rejections": reasons, "variant": VARIANT, "spp": spp,
           "render_time_s": round(time.time() - t0, 2), "resolution": [W, H],
           "plate_like": item["plate"],
           "camera": {**cparams, "K": cam["K"].tolist(), "R": cam["R"].tolist(), "t": cam["t"].tolist(),
                      "origin": cam["origin"].tolist(), "target": cam["target"].tolist()},
           "lights": linfo, "material": {**tinfo, "roughness": rough}, "background": str(bg),
           "exposure": {"target_srgb": target, "scale": k, "saturated_frac": sat},
           "coverage": cover, "shading_contrast": contrast, "effects": einfo,
           "final_image_qa": final_qa,
           "image_metrics": image_metrics,
           "benchmark_mode": BENCHMARK_MODE,
           "run_id": RUN_ID,
           "chars": chars_px, "columns": cols_px, "patches": patches,
           "pixel_convention": "liên tục: mép trên-trái=(0,0), tâm pixel (i,j)=(i+0.5,j+0.5)"}
# MỚI
    jpath.write_text(json.dumps(rec, ensure_ascii=False, indent=2), encoding="utf-8")
    return rec

def run(items, n=None, **kw):
    from tqdm.auto import tqdm
    n = n or cfg.n_images
    for it in items:
        for i in tqdm(range(n), desc=it["tag"]):
            call_kw = dict(kw)
            call_kw.setdefault("data_profile", get_data_profile(it["tag"], i, n))
            call_kw.setdefault("detail_level", get_detail_level(it["tag"], i, n))
            render_one(i, it, **call_kw)


## 14. Tổng hợp đầu ra
Dựng lại `manifest.csv` và `rec_gt.txt` từ các JSON — chạy lại nhiều lần không bị trùng dòng.

In [ ]:
def finalize_exports():
    """Dựng lại manifest.csv và rec_gt.txt từ các JSON (không bị trùng khi chạy lại)."""
    recs = [json.loads(p.read_text(encoding="utf-8")) for p in sorted((ROOT / "labels").glob("*.json"))]
    acc = [r for r in recs if r.get("accepted")]
    with open(ROOT / "manifest.csv", "w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["run_id", "sample_id", "board_id", "mesh_id", "assigned_condition", "annotation_version", "image", "object", "data_profile", "detail_level", "generation_profile",
                    "tilt_deg", "azim_deg", "fov_deg", "key_elev_deg", "fill_ratio",
                    "background", "exposure_target", "saturated_frac", "coverage", "shading_contrast",
                    "text_sharpness", "jpeg_quality", "n_visible_chars", "render_time_s"])
        for r in acc:
            w.writerow([r.get("run_id"),r.get("sample_id"),r.get("board_id"),r.get("mesh_id"),r.get("assigned_condition"),r.get("annotation_version"),r["image"], r["object"], r.get("data_profile", "unknown"),
                        r.get("detail_level", "sharp"), r.get("generation_profile", "unknown"),
                        round(r["camera"]["tilt_deg"], 2), round(r["camera"]["azim_deg"], 2),
                        round(r["camera"]["fov_deg"], 2), round(r["lights"]["key_elev_deg"], 2),
                        round(r["lights"]["fill_ratio"], 3), r["background"],
                        round(r["exposure"]["target_srgb"], 3), round(r["exposure"]["saturated_frac"], 4),
                        round(r["coverage"], 3), r["shading_contrast"],
                        r.get("final_image_qa", {}).get("text_sharpness_laplacian_var"),
                        r["effects"]["jpeg_quality"],
                        sum(c["visible"] and not c.get("pseudo") for c in r["chars"]), r["render_time_s"]])
    lines = [f"{p['file']}\t{p['text']}" for r in acc for p in r.get("patches", [])]
    (ROOT / "rec_gt.txt").write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")
    return recs, acc, lines

In [ ]:
def export_quality_splits(records):
    """
    Giữ cấu trúc cũ quality_splits/<profile>/...
    Đồng thời xuất thêm quality_level_splits/<profile>/<sharp|soft>/...
    để dễ dùng/kiểm tra hai level bên trong từng profile.
    """
    profile_root = ROOT / "quality_splits"
    level_root = ROOT / "quality_level_splits"

    for split_root in (profile_root, level_root):
        if split_root.exists():
            shutil.rmtree(split_root)

    def init_split(base):
        for sub in ("images", "patches", "labels", "overlays"):
            (base / sub).mkdir(parents=True, exist_ok=True)

    for profile in PROFILE_WEIGHTS:
        init_split(profile_root / profile)
        for level in PROFILE_LEVELS:
            init_split(level_root / profile / level)

    profile_counts = {
        profile: {"images": 0, "patches": 0}
        for profile in PROFILE_WEIGHTS
    }
    level_counts = {
        (profile, level): {"images": 0, "patches": 0}
        for profile in PROFILE_WEIGHTS for level in PROFILE_LEVELS
    }

    def copy_record_to_split(r, base):
        image_rel = Path(r["image"])
        image_name = image_rel.name
        stem = image_rel.stem
        src_image = ROOT / image_rel
        if not src_image.exists():
            print("Thiếu ảnh:", src_image)
            return 0, 0

        shutil.copy2(src_image, base / "images" / image_name)
        src_overlay = ROOT / "overlays" / f"{stem}.jpg"
        if src_overlay.exists():
            shutil.copy2(src_overlay, base / "overlays" / src_overlay.name)

        split_patches = []
        rec_gt_lines = []
        n_patches = 0
        for p in r.get("patches", []):
            src_patch = ROOT / p["file"]
            if not src_patch.exists():
                print("Thiếu patch:", src_patch)
                continue
            patch_name = src_patch.name
            shutil.copy2(src_patch, base / "patches" / patch_name)
            patch_rel = f"patches/{patch_name}"
            split_patches.append({**p, "file": patch_rel})
            rec_gt_lines.append(f"{patch_rel}\t{p['text']}")
            n_patches += 1

        split_record = {
            **r,
            "image": f"images/{image_name}",
            "patches": split_patches,
        }
        (base / "labels" / f"{stem}.json").write_text(
            json.dumps(split_record, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
        with open(base / "rec_gt.txt", "a", encoding="utf-8") as f:
            for line in rec_gt_lines:
                f.write(line + "\n")
        return 1, n_patches

    for r in records:
        if not r.get("accepted"):
            continue
        profile = r.get("data_profile", "normal")
        level = r.get("detail_level", "sharp")
        if profile not in PROFILE_WEIGHTS:
            raise ValueError(f"Profile không hợp lệ trong JSON: {profile}")
        if level not in PROFILE_LEVELS:
            raise ValueError(f"Detail level không hợp lệ trong JSON: {level}")

        # View 1: giữ nguyên cấu trúc profile-level cũ, không phá script downstream.
        base_profile = profile_root / profile
        img_n, patch_n = copy_record_to_split(r, base_profile)
        profile_counts[profile]["images"] += img_n
        profile_counts[profile]["patches"] += patch_n

        # View 2: phân chia rõ profile × độ nét.
        base_level = level_root / profile / level
        img_n, patch_n = copy_record_to_split(r, base_level)
        level_counts[(profile, level)]["images"] += img_n
        level_counts[(profile, level)]["patches"] += patch_n

    def write_manifest(base, profile, level=None):
        with open(base / "manifest.csv", "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["image", "data_profile", "detail_level", "generation_profile", "n_patches"])
            for p in sorted((base / "labels").glob("*.json")):
                rec = json.loads(p.read_text(encoding="utf-8"))
                writer.writerow([
                    rec.get("image"),
                    rec.get("data_profile", profile),
                    rec.get("detail_level", level or "sharp"),
                    rec.get("generation_profile", "unknown"),
                    len(rec.get("patches", [])),
                ])

    for profile in PROFILE_WEIGHTS:
        write_manifest(profile_root / profile, profile)
        for level in PROFILE_LEVELS:
            write_manifest(level_root / profile / level, profile, level)

    print("\n=== QUALITY SPLITS BY PROFILE ===")
    for profile, stat in profile_counts.items():
        print(f"{profile:>6}: {stat['images']} ảnh, {stat['patches']} patch")

    print("\n=== QUALITY SPLITS BY PROFILE × DETAIL LEVEL ===")
    for (profile, level), stat in level_counts.items():
        print(f"{profile:>6}/{level:<5}: {stat['images']} ảnh, {stat['patches']} patch")

    total_patches = sum(stat["patches"] for stat in profile_counts.values())
    if total_patches == 0:
        print(
            "CẢNH BÁO: chưa có patch OCR nào. Với mesh thật, hãy cấu hình "
            "cfg.labels_path tới JSON nhãn có 'columns' (text + quad_m), "
            "hoặc đặt cfg.mesh_source = 'synthetic' để kiểm tra pipeline "
            "bằng dữ liệu có nhãn giả lập."
        )
    print("Profile output:", profile_root)
    print("Profile × level output:", level_root)


## 15. Kiểm thử
Mỗi dòng PASS/FAIL dưới đây đến từ một phép kiểm tra **thật sự chạy**. Không có dòng PASS nào được in sẵn.

In [ ]:
CHECKS = {}   # tên kiểm tra -> (PASS/FAIL, chi tiết). Chỉ được ghi PASS khi assert thật sự chạy.

def record(name, ok, detail=""):
    CHECKS[name] = ("PASS" if ok else "FAIL", detail)
    print(("PASS" if ok else "FAIL"), "|", name, "|", detail)

### 15.1. Chuyển hộp trên depth map → tứ giác 3D
Dùng khi nạp nhãn thật từ dữ liệu SiNoC sau này.

In [ ]:
def test_depthmap_bbox_to_quads():
    # P_depth đúng quy ước của bước sinh depth map: gốc pixel trên-trái, y ảnh hướng xuống
    xmin, ymax, sx, sy = -150.0, 100.0, 4.0, 4.0            # mm, px/mm
    P = np.array([[sx, 0, -xmin * sx], [0, -sy, ymax * sy], [0, 0, 1.0]])
    pts_mm = np.array([[-120.0, 80.0], [-100.0, 55.0]])      # hai góc hộp trên mặt ván
    px = (P @ np.c_[pts_mm, np.ones(2)].T).T[:, :2]
    q = depthmap_bbox_to_quads([[px[0, 0], px[0, 1], px[1, 0], px[1, 1]]], P, surface_z=0.0)[0]
    exp = np.array([[-0.120, 0.080, 0], [-0.100, 0.080, 0], [-0.100, 0.055, 0], [-0.120, 0.055, 0]])
    err = float(np.abs(q - exp).max())
    record("depthmap_bbox_to_quads", err < 1e-9, f"sai số lớn nhất {err:.2e} m")

test_depthmap_bbox_to_quads()

### 15.2. Chuẩn hóa tư thế
Ván giả được xoay/dời ngẫu nhiên rồi chuẩn hóa lại. Yêu cầu: pháp tuyến lệch < 0.5°, chiều cao chữ đúng ±0.1 mm.

In [ ]:
print("Font từ Config:",cfg.font_path)


In [ ]:
def test_pose_normalization(n_trials=3):
    """Ván giả + tư thế ngẫu nhiên -> chuẩn hóa -> pháp tuyến < 0.5°, chiều cao chữ đúng ±0.1 mm."""
    old = cfg.grid_res_mm
    cfg.grid_res_mm = max(old, 0.5)                         # lưới thô cho nhanh, đủ để kiểm
    try:
        mesh, ch, co = make_synthetic_block()
    finally:
        cfg.grid_res_mm = old
    rng = np.random.default_rng(99); worst_ang, worst_rel = 0.0, 0.0
    for _ in range(n_trials):
        ax = rng.normal(size=3); ax /= np.linalg.norm(ax)
        ang = rng.uniform(10, 170)
        R = trimesh.transformations.rotation_matrix(math.radians(ang), ax)
        R[:3, 3] = rng.uniform(-0.3, 0.3, 3)
        mb = mesh.copy(); mb.apply_transform(R)
        mn, M, plate, info = normalize_pose(mb)
        assert plate, "ván giả phải được nhận là dạng tấm"
        n_rec = M[:3, :3] @ (R[:3, :3] @ np.array([0, 0, 1.0]))
        worst_ang = max(worst_ang, math.degrees(math.acos(np.clip(n_rec[2], -1, 1))))
        V = np.asarray(mn.vertices); top = V[V[:, 2] > -0.01, 2]
        h, e = np.histogram(top, bins=400); floor = 0.5 * (e[h.argmax()] + e[h.argmax() + 1])
        rel = (np.quantile(top, 0.999) - floor) * 1000
        worst_rel = max(worst_rel, abs(rel - cfg.relief_mm))
    record("chuẩn hóa tư thế", worst_ang < 0.5 and worst_rel < 0.1,
           f"sai lệch pháp tuyến lớn nhất {worst_ang:.4f}°, sai lệch chiều cao chữ lớn nhất {worst_rel:.3f} mm "
           f"qua {n_trials} tư thế ngẫu nhiên")

if cfg.mesh_source=='synthetic' or (cfg.font_path and Path(cfg.font_path).exists()):
    test_pose_normalization()
    globals().pop('SYNTHETIC_MASKS',None)
else:
    CHECKS['synthetic pose selftest']=('SKIP','No local font; actual-mesh projection test still runs.')
    print('SKIP synthetic pose test: configure a font to enable it.')

## 16. Nạp vật thể theo cấu hình

In [ ]:
RAW_ITEMS = prepare_mesh_items()
ITEMS = [setup_item(*x) for x in RAW_ITEMS]
for it in ITEMS:
    ext = it["mesh"].extents * 1000
    record(f"nạp mesh [{it['tag']}]", bool(np.all(ext > 1) and np.all(ext < 5000)),
           f"kích thước {np.round(ext, 1)} mm, plate_like={it['plate']}")

In [ ]:
for it in ITEMS:
    print(it['tag'],"| stroke:",it.get('stroke_source'),"| validated:",it.get('stroke_validated'),
          "| ROI validated:",it.get('roi_validated'),"| annotation errors:",it.get('annotation_errors'))


### 16.1. Kiểm tra phép chiếu — cell quyết định
Notebook **dừng** nếu sai số ≥ 1.5 px. Nếu dừng, mở ảnh trong `output/selftest/`.

In [ ]:
for it in ITEMS:
    err, rep = projection_selftest(it["mesh"], spp=8 if VARIANT.startswith("cuda") else 4)
    for r in rep:
        print("   ", r)
    record(f"phép chiếu [{it['tag']}]", err < 1.5, f"sai số trung bình {err:.3f} px (ngưỡng 1.5)")
    assert err < 1.5, "Phép chiếu sai — DỪNG. Xem ảnh trong output/selftest/"

## 17 Render pilot và xuất dữ liệu nghiên cứu

Giữ assigned_condition. Không phân loại hậu kỳ để đổi điều kiện thí nghiệm. Research mode bỏ gate độ nét/contrast,
giữ coverage/clipping và ghi retry. Profile mode: tỷ lệ mục tiêu CLEAN/NORMAL/HARD=50/25/25;
controlled mode dùng danh sách condition có ID và seed theo repetition.


In [ ]:
print("Số ảnh mỗi vật thể:",cfg.n_images,"| experiment:",cfg.experiment_mode)


In [ ]:
# Tắt xóa output tự động. Mỗi run dùng một thư mục riêng.
print("Giữ nguyên dữ liệu trong:", ROOT)


In [ ]:
# Phân loại profile hậu kỳ dựa trên các JSON và ảnh JPG đã render.
# Không render lại hoặc sửa pixel ảnh; chỉ cập nhật data_profile và tạo báo cáo chỉ số.
import csv
import json
import math
from pathlib import Path


def _float_or(value, default):
    try:
        value = float(value)
        return value if math.isfinite(value) else default
    except (TypeError, ValueError):
        return default


def _angle_delta_deg(a, b):
    """Khoảng cách góc nhỏ nhất trong [0, 180]."""
    if a is None or b is None:
        return 180.0
    try:
        return abs((float(a) - float(b) + 180.0) % 360.0 - 180.0)
    except (TypeError, ValueError):
        return 180.0


def _image_text_sharpness(image_path, rec):
    """Variance of Laplacian trên polygon ký tự nhìn thấy; cao hơn thường nét hơn.
    Dùng JSON chars[].poly_px để đo đúng vùng có chữ, tránh đo cả nền gỗ.
    """
    image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        return None, "unavailable"
    H, W = image.shape[:2]
    lap = cv2.Laplacian(image, cv2.CV_32F, ksize=3)
    chars = [c for c in rec.get("chars", [])
             if c.get("visible") and not c.get("pseudo") and c.get("poly_px")]

    # Giới hạn số polygon để chạy ổn định kể cả JSON có nhiều ký tự.
    if len(chars) > 256:
        ids = np.linspace(0, len(chars) - 1, 256).astype(int)
        chars = [chars[k] for k in ids]

    vals = []
    for ch in chars:
        try:
            poly = np.asarray(ch["poly_px"], dtype=np.float32).reshape(-1, 2)
            if len(poly) < 3 or not np.isfinite(poly).all():
                continue
            x0 = max(0, int(math.floor(float(poly[:, 0].min()))))
            y0 = max(0, int(math.floor(float(poly[:, 1].min()))))
            x1 = min(W, int(math.ceil(float(poly[:, 0].max()))) + 1)
            y1 = min(H, int(math.ceil(float(poly[:, 1].max()))) + 1)
            if x1 - x0 < 4 or y1 - y0 < 4:
                continue
            local_poly = np.round(poly - np.array([x0, y0], dtype=np.float32)).astype(np.int32)
            mask = np.zeros((y1 - y0, x1 - x0), dtype=np.uint8)
            cv2.fillPoly(mask, [local_poly], 255)
            vals_px = lap[y0:y1, x0:x1][mask > 0]
            if vals_px.size >= 12:
                v = float(np.var(vals_px))
                if math.isfinite(v):
                    vals.append(v)
        except (ValueError, TypeError, IndexError):
            continue

    if vals:
        return float(np.median(vals)), "visible_character_regions"

    # Không có nhãn ký tự usable: phép đo fallback toàn ảnh chỉ để phân hạng tương đối.
    small = cv2.resize(image, (max(8, W // 2), max(8, H // 2)), interpolation=cv2.INTER_AREA)
    fallback = cv2.Laplacian(small, cv2.CV_32F, ksize=3)
    return float(np.var(fallback)), "whole_image_fallback"


def _profile_metrics(rec, root):
    camera = rec.get("camera") or {}
    lights = rec.get("lights") or {}
    effects = rec.get("effects") or {}
    noise = effects.get("noise") or {}
    motion = effects.get("motion_blur")
    enhance = effects.get("enhance") or {}

    tilt = _float_or(camera.get("tilt_deg"), 1e6)
    cam_az = camera.get("azim_deg")
    light_az = lights.get("key_azim_deg")
    light_delta = _float_or(lights.get("camera_axis_offset_deg"), _angle_delta_deg(cam_az, light_az))
    light_obliqueness = abs(math.sin(math.radians(light_delta)))
    key_power = _float_or(lights.get("key_power_multiplier"), 1.0)
    key_elev = _float_or(lights.get("key_elev_deg"), 0.0)
    fill = _float_or(lights.get("fill_ratio"), 0.0)
    vignette = _float_or(effects.get("vignette"), 1.0)
    a = _float_or(noise.get("a"), 1.0)
    b = _float_or(noise.get("b"), 1.0)

    image_rel = rec.get("image")
    image_path = (root / image_rel) if image_rel else None
    sharpness = None
    sharpness_source = "unavailable"
    image_mean = 0.5
    exposure = rec.get("exposure") or {}
    exposure_target = _float_or(exposure.get("target_srgb"), 0.0)
    shading_contrast = _float_or(rec.get("shading_contrast"), 0.0)
    if image_path is not None and image_path.exists():
        sharpness, sharpness_source = _image_text_sharpness(image_path, rec)
        try:
            gray = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
            if gray is not None:
                image_mean = float(gray.mean()) / 255.0
        except Exception:
            pass

    if sharpness is None:
        # Chỉ dùng khi thiếu ảnh hoặc không đọc được ảnh; lớn hơn = giả định nét hơn.
        defocus = _float_or(effects.get("defocus_sigma_px"), 1.0)
        motion_len = _float_or((motion or {}).get("len_px"), 0.0)
        sharpness = 1.0 / (1.0 + defocus + 0.5 * motion_len)
        if not enhance.get("skipped"):
            sharpness += 0.05 * _float_or(enhance.get("unsharp_amount"), 0.0)
        sharpness_source = "effects_proxy"

    # Ước lượng độ lệch chuẩn nhiễu từ thông số camera-effects trong JSON.
    noise_sigma = math.sqrt(max(0.0, a * image_mean + b))
    # Ánh sáng tạo bóng: đèn lệch hướng camera và độ cao thấp làm điểm số tăng.
    shadow_score = 0.65 * light_obliqueness + 0.35 * (1.0 - min(max(key_elev, 0.0), 90.0) / 90.0)
    # Giá trị thấp = ít fill light / góc tối mạnh hơn.
    ambient_score = fill - 0.05 * max(0.0, vignette)

    return {
        "tilt_deg": tilt,
        "tilt_bin_2deg": int(math.floor(tilt / 2.0)) if tilt < 1e5 else 10**9,
        "text_sharpness_laplacian_var": float(sharpness),
        "sharpness_source": sharpness_source,
        "noise_sigma_estimate": float(noise_sigma),
        "camera_light_azimuth_delta_deg": float(light_delta),
        "camera_axis_offset_deg": float(light_delta),
        "key_power_multiplier": float(key_power),
        "light_obliqueness_score": float(light_obliqueness),
        "light_shadow_score": float(shadow_score),
        "key_elev_deg": float(key_elev),
        "fill_ratio": float(fill),
        "vignette": float(vignette),
        "ambient_light_score": float(ambient_score),
        "shading_contrast": float(shading_contrast),
        "exposure_target_srgb": float(exposure_target),
        "image_mean_srgb": float(image_mean),
    }


def reclassify_profiles_from_json(root=None):
    """Xếp hạng JSON theo từng vật thể; clean 50%, hard 25%, normal phần còn lại.

    Ưu tiên clean: góc nhỏ -> nét -> ít nhiễu -> đèn gần hướng camera -> fill tốt.
    Ưu tiên hard: góc lớn -> mờ -> ánh sáng gây bóng -> nhiễu -> fill thấp.
    """
    root = Path(root or ROOT)
    labels_dir = root / "labels"
    json_paths = sorted(labels_dir.glob("*.json"))
    if not json_paths:
        raise FileNotFoundError(f"Không tìm thấy JSON ảnh trong {labels_dir}")

    grouped = {}
    for jp in json_paths:
        try:
            rec = json.loads(jp.read_text(encoding="utf-8"))
        except Exception as exc:
            print(f"[WARN] Bỏ qua JSON không đọc được {jp.name}: {exc}")
            continue
        if not rec.get("accepted") or not rec.get("image"):
            continue
        # Preserve the profile used during rendering for auditing/reproducibility.
        if "generation_profile" not in rec:
            rec["generation_profile"] = rec.get("data_profile", "unknown")
        metrics = _profile_metrics(rec, root)
        rec["profile_selection"] = {
            "method": "v3_dual_detail_levels_raking_light_final_image_qa",
            "priority_clean": [
                "rendered as CLEAN; plate tilt <= 2 degrees",
                "zero noise, no motion blur, no vignette",
                "raking-light shading contrast and final-JPEG text sharpness both pass",
                "bright exposure target and strong fill light",
            ],
            "priority_hard": [
                "rendered as HARD; plate tilt >= 20 degrees",
                "low key power and low fill",
                "retain a minimum readable-text floor",
            ],
            "metrics": metrics,
        }
        jp.write_text(json.dumps(rec, ensure_ascii=False, indent=2), encoding="utf-8")
        obj = str(rec.get("object") or jp.stem.rsplit("_", 1)[0])
        grouped.setdefault(obj, []).append({"path": jp, "record": rec, "metrics": metrics})

    report_rows = []
    total_counts = {p: 0 for p in PROFILE_WEIGHTS}
    if not grouped:
        raise RuntimeError("Không có JSON accepted có ảnh để phân loại profile.")

    for obj, items in sorted(grouped.items()):
        n = len(items)
        clean_n = min(n, int(math.floor(n * PROFILE_WEIGHTS["clean"] + 0.5)))
        hard_n = min(n - clean_n, int(math.floor(n * PROFILE_WEIGHTS["hard"] + 0.5)))
        for it in items:
            m = it["metrics"]
            rec = it["record"]
            effects = rec.get("effects") or {}
            noise = effects.get("noise") or {}
            motion = effects.get("motion_blur")
            plate = bool(rec.get("plate_like", False))
            generation = rec.get("generation_profile", "unknown")
            rendered_profile = rec.get("render_profile", rec.get("data_profile", "unknown"))
            detail_level = rec.get("detail_level", "sharp")
            if detail_level not in PROFILE_LEVELS:
                detail_level = "sharp"
            clean_requirements = get_profile_config("clean", detail_level)
            max_clean_defocus = max(clean_requirements["camera_effects"]["defocus_sigma_px"])
            min_clean_sharpness = float(clean_requirements["qa_min_text_sharpness"])
            min_clean_contrast = float(clean_requirements["qa_min_shading_contrast"])
            clean_eligible = (
                rendered_profile == "clean"
                and generation.startswith("clean_")
                and (not plate or m["tilt_deg"] <= 2.0001)
                and abs(_float_or(noise.get("a"), 1.0)) <= 1e-12
                and abs(_float_or(noise.get("b"), 1.0)) <= 1e-12
                and _float_or(effects.get("defocus_sigma_px"), 1e6) <= max_clean_defocus + 1e-6
                and not motion
                and _float_or(effects.get("vignette"), 1.0) <= 1e-6
                and m["text_sharpness_laplacian_var"] >= min_clean_sharpness
                and m["shading_contrast"] >= min_clean_contrast
                and m["exposure_target_srgb"] >= 0.45
                and m["fill_ratio"] >= 0.10
                and 8.0 <= m["key_elev_deg"] <= 24.0
            )
            hard_eligible = (
                rendered_profile == "hard"
                and generation.startswith("hard_")
                and (not plate or m["tilt_deg"] >= 20.0)
                and m["key_power_multiplier"] <= 0.55
                and m["fill_ratio"] <= 0.03
            )
            it["clean_eligible"] = clean_eligible
            it["hard_eligible"] = hard_eligible
            it["clean_key"] = (
                0 if clean_eligible else 1,
                -m["text_sharpness_laplacian_var"],
                m["noise_sigma_estimate"],
                -m["fill_ratio"],
                it["path"].name,
            )
            it["hard_key"] = (
                0 if hard_eligible else 1,
                -m["tilt_deg"],
                m["key_power_multiplier"],
                m["fill_ratio"],
                -m["noise_sigma_estimate"],
                -m["vignette"],
                it["path"].name,
            )

        # Không ép ảnh vi phạm điều kiện hình học/nhiễu vào CLEAN/HARD chỉ để đủ quota.
        clean_pool = [x for x in items if x["clean_eligible"]]
        clean_selected = sorted(clean_pool, key=lambda x: x["clean_key"])[:clean_n]
        clean_ids = {id(x) for x in clean_selected}
        remaining = [x for x in items if id(x) not in clean_ids]
        hard_pool = [x for x in remaining if x["hard_eligible"]]
        hard_selected = sorted(hard_pool, key=lambda x: x["hard_key"])[:hard_n]
        hard_ids = {id(x) for x in hard_selected}

        for it in items:
            rec, m, jp = it["record"], it["metrics"], it["path"]
            if id(it) in clean_ids:
                final_profile = "clean"
            elif id(it) in hard_ids:
                final_profile = "hard"
            else:
                final_profile = "normal"
            rec["observed_quality"] = final_profile
            rec["profile_selection"]["final_profile"] = final_profile
            rec["profile_selection"]["quota_for_object"] = {
                "accepted_images": n,
                "target": {"clean": clean_n, "hard": hard_n, "normal": n - clean_n - hard_n},
                "actual": {"clean": len(clean_ids), "hard": len(hard_ids), "normal": n - len(clean_ids) - len(hard_ids)},
                "constraints_were_prioritized_over_quota": True,
            }
            jp.write_text(json.dumps(rec, ensure_ascii=False, indent=2), encoding="utf-8")
            total_counts[final_profile] += 1
            report_rows.append({
                "object": obj,
                "image": rec.get("image"),
                "json": jp.name,
                "generation_profile": rec.get("generation_profile", "unknown"),
                "render_profile": rec.get("render_profile", rec.get("data_profile", "unknown")),
                "detail_level": rec.get("detail_level", "sharp"),
                "final_profile": final_profile,
                **m,
            })

    report_path = root / "profile_selection_report.csv"
    report_fields = ["object", "image", "json", "generation_profile", "render_profile", "detail_level", "final_profile",
                     "tilt_deg", "tilt_bin_2deg", "text_sharpness_laplacian_var", "sharpness_source",
                     "noise_sigma_estimate", "camera_light_azimuth_delta_deg", "camera_axis_offset_deg",
                     "key_power_multiplier", "light_obliqueness_score", "light_shadow_score", "key_elev_deg",
                     "fill_ratio", "vignette", "ambient_light_score", "shading_contrast",
                     "exposure_target_srgb", "image_mean_srgb"]
    with report_path.open("w", newline="", encoding="utf-8-sig") as fh:
        writer = csv.DictWriter(fh, fieldnames=report_fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(sorted(report_rows, key=lambda x: (x["object"], x["final_profile"], x["image"] or "")))

    print("=== Phân loại lại từ JSON ===")
    print("Số ảnh accepted theo profile:", total_counts)
    print("Báo cáo chi tiết:", report_path)
    print("Lưu ý: observed_quality chỉ là chẩn đoán; assigned_condition và data_profile lúc render không bị thay đổi.")
    for obj, one_group in sorted(grouped.items()):
        obj_clean = sum(1 for x in one_group if x.get("clean_eligible"))
        obj_hard = sum(1 for x in one_group if x.get("hard_eligible"))
        if obj_clean < int(math.floor(len(one_group) * PROFILE_WEIGHTS["clean"] + 0.5)):
            print(f"[WARN] {obj}: không đủ ảnh CLEAN vượt đồng thời góc camera, raking contrast, độ nét và noise=0; giữ phần còn lại ở NORMAL.")
        if obj_hard < int(math.floor(len(one_group) * PROFILE_WEIGHTS["hard"] + 0.5)):
            print(f"[WARN] {obj}: không đủ ảnh HARD đạt hard gate; tỷ lệ thực tế có thể lệch 50/25/25.")
    return report_rows



In [ ]:
required = ["cfg", "ROOT", "PROFILE_WEIGHTS", "PROFILE_CONFIGS", "ITEMS",
            "render_one", "benchmark_render_metrics", "BENCHMARK_MODE", "RUN_ID",
            "get_data_profile", "get_detail_level", "finalize_exports",
            "export_quality_splits", "reclassify_profiles_from_json"]
missing = [key for key in required if key not in globals()]
if missing:
    raise RuntimeError("Chưa chạy các cell phía trên: " + ", ".join(missing)
                       + ". Chạy notebook từ phần 1 đến đây theo thứ tự.")

# Render hàng loạt + xuất 3 profile OCR: clean 50% / normal 25% / hard 25%.
print("Tỷ lệ profile OCR mục tiêu:", PROFILE_WEIGHTS)
print("Mỗi profile có hai level: sharp / soft; lịch level cân bằng bên trong từng profile.")
print("Benchmark:", BENCHMARK_MODE, "| gate tương phản/độ nét bật:", not BENCHMARK_MODE)
print("HARD giữ nguyên dải camera/ánh sáng/noise nền; level chỉ phân tầng độ nét hậu kỳ.")

def run_verbose(items, n=None):
    total_n = n or cfg.n_images
    for it in items:
        for i in range(total_n):
            profile,level,condition_id,_ = experiment_assignment(it["tag"],i,total_n)
            t = time.time()
            r = render_one(i, it, data_profile=profile, detail_level=level)
            if r is None:
                print(f"[{i}] {profile}/{level} exists, skipped", flush=True)
                continue
            w = r.get("material", {}).get("wear", {})
            print(
                f"[{i}] condition={condition_id} profile={profile}/{level} {time.time()-t:.1f}s "
                f"attempts={r.get('attempts')} rejected={r.get('rejections')} "
                f"wear={w.get('time_s')}s RAM free={ram_free_gb():.1f}GB",
                flush=True,
            )

t_run = time.time()
run_verbose(ITEMS)
if not BENCHMARK_MODE:
    print("Research mode preserves assigned profiles; reclassification is opt-in diagnostic only.")
recs, acc, rec_lines = finalize_exports()
export_quality_splits(acc)

research_summary=summarize_run(ROOT,{
    "run_id":RUN_ID,"config":asdict(cfg),"profiles":PROFILE_CONFIGS,
    "metric_config":asdict(METRIC_CONFIG),"n_requested":cfg.n_images*len(ITEMS),
    "n_accepted":len(acc),"n_rejected":len(recs)-len(acc),
    "rejected_samples":[{"sample_id":r.get('sample_id'),"reasons":r.get('rejections')} for r in recs if not r.get('accepted')],
})
profile_counts = {
    p: sum(1 for r in acc if r.get("data_profile") == p)
    for p in PROFILE_WEIGHTS
}
patch_counts = {
    p: sum(len(r.get("patches", [])) for r in acc if r.get("data_profile") == p)
    for p in PROFILE_WEIGHTS
}
detail_counts = {
    p: {
        level: sum(1 for r in acc
                   if r.get("data_profile") == p and r.get("detail_level", "sharp") == level)
        for level in PROFILE_LEVELS
    }
    for p in PROFILE_WEIGHTS
}
print("Ảnh được nhận theo profile:", profile_counts)
print("Ảnh được nhận theo profile × level:", detail_counts)
print("Patch OCR theo profile:", patch_counts)
print(
    f"Xong trong {time.time() - t_run:.0f} s: "
    f"{len(acc)} ảnh được nhận / {len(recs)} lượt"
)


### 17.1. Xuất CSV metric
Chạy sau render hàng loạt. Các ảnh bị reject có trạng thái `not_measured`.


In [ ]:
# Rebuild metric tables from authoritative per-sample logs. No metrics gate affects acceptance.
research_summary=summarize_run(ROOT,{"run_id":RUN_ID,"config":asdict(cfg),"profiles":PROFILE_CONFIGS,
    "n_requested":cfg.n_images*len(ITEMS),"n_accepted":len(acc),"n_rejected":len(recs)-len(acc)})
import pandas as pd
df_metrics=pd.read_csv(ROOT/"render_metrics.csv")
display(df_metrics)
print("Measured samples:",research_summary['n_measured'],"| valid E_net:",research_summary['n_valid_E_net'])
print("Invalid samples are kept with status/reason; CER remains pending until collaborator handoff.")


## 18. Bảng kiểm tra chất lượng
Lưới overlay ngẫu nhiên và phân bố các tham số đã dùng.

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

def qa_dashboard(acc, recs, n_show=12, seed=0):
    rs = np.random.default_rng(seed)
    pick = [acc[k] for k in rs.choice(len(acc), min(n_show, len(acc)), replace=False)] if acc else []
    if pick:
        cols = 4; rows = math.ceil(len(pick) / cols)
        fig, axs = plt.subplots(rows, cols, figsize=(20, 3.8 * rows))
        for ax in np.ravel(axs):
            ax.axis("off")
        for ax, r in zip(np.ravel(axs), pick):
            ov = cv2.cvtColor(cv2.imread(str(ROOT / "overlays" / Path(r["image"]).name)), cv2.COLOR_BGR2RGB)
            ax.imshow(ov)
            ax.set_title(f"tilt {r['camera']['tilt_deg']:.0f}° | đèn {r['lights']['key_elev_deg']:.0f}° | "
                         f"nền {r['background']}", fontsize=9)
        plt.tight_layout(); plt.savefig(ROOT / "qa_overlays.jpg", dpi=80); plt.show()
    fig, axs = plt.subplots(1, 5, figsize=(22, 3.5))
    axs[0].hist([r["camera"]["tilt_deg"] for r in acc], 15); axs[0].set_title("Góc nghiêng camera (°)")
    axs[1].hist([r["lights"]["key_elev_deg"] for r in acc], 15); axs[1].set_title("Độ cao đèn chính (°)")
    axs[2].hist([r["exposure"]["target_srgb"] for r in acc], 15); axs[2].set_title("Mục tiêu phơi sáng")
    rej = Counter(x.split()[0] + (" " + x.split()[1] if x.startswith("low") else "")
                  for r in recs for x in r.get("rejections", []))
    axs[3].bar(list(rej.keys()) or ["(không có)"], list(rej.values()) or [0]); axs[3].set_title("Lý do loại")
    axs[3].tick_params(axis="x", rotation=20)
    axs[4].hist([r["render_time_s"] for r in acc], 15); axs[4].set_title("Thời gian mỗi ảnh (s)")
    plt.tight_layout(); plt.savefig(ROOT / "qa_stats.jpg", dpi=80); plt.show()
    if acc:
        print(f"Thời gian trung bình {np.mean([r['render_time_s'] for r in acc]):.1f} s/ảnh | "
              f"tỉ lệ nhận {len(acc)}/{len(recs)} | tổng số lần thử lại {sum(r.get('attempts', 1) - 1 for r in acc)}")

qa_dashboard(acc, recs)

## 19. Nghiệm thu

In [ ]:
n_img = len(list((ROOT / "images").glob("*.jpg")))
n_ov = len(list((ROOT / "overlays").glob("*.jpg")))
record("xuất ảnh + JSON + overlay", len(acc) > 0 and n_img == len(acc) == n_ov,
       f"{len(acc)} ảnh được nhận, {n_img} file ảnh, {n_ov} overlay, yêu cầu {cfg.n_images * len(ITEMS)}")
worst_sat = max((r["exposure"]["saturated_frac"] for r in acc), default=1.0)
record("không cháy sáng (<1% pixel vật thể)", worst_sat < cfg.max_saturated_frac,
       f"ảnh tệ nhất {100 * worst_sat:.3f}%")
if any(it["has_text"] for it in ITEMS):
    n_patch = len(list((ROOT / "patches").glob("*.jpg")))
    record("rec_gt.txt khớp số patch", len(rec_lines) == n_patch, f"{len(rec_lines)} dòng, {n_patch} patch")
print("\n================ NGHIỆM THU ================")
for k, (st, d) in CHECKS.items():
    print(f"[{st}] {k} — {d}")
print("[CẦN XEM BẰNG MẮT] khung nhãn nằm đúng trên chữ trong output/overlays và qa_overlays.jpg")
if FALLBACKS:
    print("\nPhương án dự phòng đã dùng:"); [print("  -", f) for f in FALLBACKS]
for r in acc:
    name=Path(r['image']).stem
    report=json.loads((ROOT/'logs'/f'{name}_metric.json').read_text())
    missing=[p for p in report['files'].values() if not (ROOT/p).exists()]
    assert not missing,(name,missing)
    assert report['object_fallback'] is False
    if not report.get('stroke_validated') or not report.get('roi_validated'):
        assert report['E_net'] is None
record("V5 artifact contract",True,f"{len(acc)} samples; artifacts present, no object fallback")


## 20. Nén đầu ra để tải về

In [ ]:
run_metadata={"run_id":RUN_ID,"config":asdict(cfg),"profiles":PROFILE_CONFIGS,
    "n_requested":cfg.n_images*len(ITEMS),"n_accepted":len(acc),"n_rejected":len(recs)-len(acc),
    "checks":CHECKS,"fallbacks":FALLBACKS,"metric_config":asdict(METRIC_CONFIG),
    "rejected_samples":[{"sample_id":r.get('sample_id'),"reasons":r.get('rejections')} for r in recs if not r.get('accepted')]}
summary=summarize_run(ROOT,run_metadata)
write_json(ROOT/'run_summary.json',summary)
print("Output:",ROOT)
# Optional, explicit packaging (not run automatically):
# shutil.make_archive(str(ROOT), 'zip', root_dir=ROOT)


## Giới hạn và bàn giao

- E_proxy không được thay thế E_net; proposal chưa xác thực có thể sai với ván cong, mặt lõm hoặc hư hại.
- E_net cần cả mask và ROI đã được duyệt; đủ pixel chưa chứng minh vùng đo đúng.
- Metric reference là bản thử nghiệm có version. Son's plugin được lưu riêng, không tự sửa nhãn tính hợp lệ.
- Chỉ giữ gate lỗi kỹ thuật (coverage/clipping); không loại ảnh vì E/Laplacian thấp trong research mode.
- Sáu profile là điều kiện tổng hợp nhiều yếu tố, không chứng minh riêng tác động camera/đèn/blur.
- Controlled mode: phải khóa các yếu tố còn lại bằng các khoảng [v,v]; cùng seed hỗ trợ paired comparison,
  nhưng retry có thể làm lệch cặp, cần đối chiếu camera/material/retry trong log trước phân tích.
- Ảnh render là sRGB uint8 trước camera effects, không phải HDR raw. Ảnh final là JPEG đã giải mã.
- Khung/đường kẻ synthetic được tách khỏi chữ ngay ở generator; nhãn file thật phải được Lâm duyệt.
- Giảm lưới chỉ kiểm tra khoảng cách trên patch giữa ván, chưa chứng minh bảo toàn toàn bộ nét trên ván.
- Tập ảnh một ván chưa đủ để kết luận khả năng khái quát; phân tích/chia tập theo board.
- Không có kết quả OCR hay CER giả lập. Dùng join_ocr() khi người phụ trách cung cấp kết quả có ID/config/GT version.
